# AIM:
create evaluation workflow. Taking the manually extracted Ci impacts (validation set) and compare it with the CI impacts (llm_geolocations.ipynb) extrracted by the first LLM 1. 
As a first step the evaluation should be done only for the direct CI impacts - CI type, damage and geolocation

Issue:
* What is needed an approach that recognizes when an direct impact case is not detected by the model

Idea: 
* Split the original texts passed to the model on the exact chunks as again
* Then chunkwise check if the CI impacts from the validation set correspond in number and their textual similarity to the CI impacts infered by the LLM 1 and Entity Linking 

In [158]:
# # settings for CUDA and PYTORCH
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
print(os.environ["CUDA_VISIBLE_DEVICES"])
# os.environ["CUDA_VISIBLE_DEVICES"]="0"
os.environ["PYTORCH_ALLOC_CONF"]="expandable_segments:True" ## improve memory allocation

# # settings for debugging CUDA errors (pinpoint exact line of error)
os.environ["TORCH_USE_CUDA_DSA"] = "1"
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1" 

# activate global venv explicitly
os.environ["VIRTUAL_ENV"] = "/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/.venv"


import torch

print(torch.cuda.is_available())
print(torch.cuda.device_count())  # should give 2
print(torch.cuda.get_device_name())
print(torch.cuda.get_device_properties(0))
# print(torch.cuda.get_device_properties(1))
print(torch.cuda.get_device_capability())
print(torch.cuda.get_arch_list())
print(torch.__version__)
print(torch.version.cuda)


0
True
1
NVIDIA A100-PCIE-40GB
_CudaDeviceProperties(name='NVIDIA A100-PCIE-40GB', major=8, minor=0, total_memory=40441MB, multi_processor_count=108, uuid=49cb0d59-8657-8b70-8dd2-c1db7aab24a4, pci_bus_id=131, pci_device_id=0, pci_domain_id=0, L2_cache_size=40MB)
(8, 0)
['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
2.8.0+cu128
12.8


In [159]:
import os
import sys
from pathlib import Path
import io
import gc
import time
import warnings
import subprocess
import importlib
import glob

from unidecode import unidecode
import langdetect
from fuzzywuzzy import fuzz
import torch
from huggingface_hub import login
import numpy as np
import spacy
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from matplotlib import pyplot as plt


sys.path.append('./')
from src.settings import settings as s
import src.document_cleaning as dc
import src.translation_model as tm
import src.utils as u
import src.datahandler as dh
import src.postprocess as pp

# login to HF
# NOTE raises exception when env.variable does not exist (compared to os.envrion.get and its shortcut os.getenv)
os.getenv("HUGGINGFACE_TOKEN")

#  automatic linebreaks and multi-line cells.
pd.set_option("display.colheader_justify", "left")
pd.set_option('display.max_colwidth', 5000)


step = "step2"


### Direct CI impacts: LLM 1 vs domain-expertise 

In [160]:
#  Suppress future warnings from PyTorch
warnings.filterwarnings("ignore", category=FutureWarning)


#  Define data dir where tags.csv and domain-expertise derived tag lists are found 
# VALID_DATA_FILENAME = s.VALID_DATA_FILENAME
VALID_DATA_FILENAME = "table_ci_impacts_sm.csv"
PATH_VALID_DATA = s.PATH_VALID_DATA
PATH_EVAL_RESULT = s.PATH_EVAL_RESULT

# s.LLM_DATA_FILENAME = "llm1_geollm_step2_Koks 2022.csv" #f"df_responses_{step}_ner_geollm.csv"
# LLM_DATA_FILEPATH = Path("interim_results" / s.LLM_DATA_FILENAME)
# LLM_DATA_FILEPATH = Path(s.PATH_LLM_DATA / s.LLM_DATA_FILENAME)
SIMILARITY_LLM_FILENAME = s.SIMILARITY_LLM_FILENAME

df_valid_org = pd.read_csv(
    PATH_VALID_DATA / VALID_DATA_FILENAME,
    # usecols=["publication_id", "ci1_type", "ci1_damage", "ci1_location", "sentence_reference"],
)
print(len(df_valid_org))
## pre-process: 
# remove undone entries
df_valid_org = df_valid_org[~df_valid_org.astype(str).apply(lambda x: x.str.contains("xx")).any(axis=1)]
# remove further location info (e.g. that entry is a town, Landkreis, Bavaria etc.)
# df_valid_org["ci1_location"] = df_valid_org["ci1_location"].replace(r"\s*\(.*\)", "", regex=True).str.strip()
df_valid_org = df_valid_org.dropna(subset=["publication_id"], how="all") # drop rows where citation info is missing
print(len(df_valid_org))
print(df_valid_org.publication_id.unique())

print(f"Collecting LLM responses from {step}")
# ## prediction data
# df_pred = pd.read_csv(
#     LLM_DATA_FILEPATH,
#    # usecols=["citation_id", "chunk_id", "infrastructure_type", "damage", "location", "chunk_text"]
# )
df_pred = pd.DataFrame()
for i, file in enumerate(glob.glob(os.path.join("./interim_results", f'*{step}*.csv'))):
#for i, file in enumerate(glob.glob(os.path.join(s.PATH_DATA, "llm_outputs/responses_single_docs", f'*{step}*.csv'))):
    print(i, file)
    df_pred = pd.concat([df_pred, pd.read_csv(file)], ignore_index=True)



165
142
<ArrowStringArray>
[                     'ABC 2024',                  'Artemis 2015',
                    'Brown 2010',            'Containerlift 2024',
                   'Deidda 2025',                      'EFE 2024',
                 'Euronews 2024', 'European Investment Bank 2025',
                  'Ferlita 2023',        'Gilbody Dickerson 2024',
              'Karakatsani 2023',                     'Kaur 2025',
                   'Keller 2014',                   'Khazai 2013',
                     'Koks 2022',              'Lloyds List 2024',
                      'PWC 2015',                'Rozendaal 2021',
                'Skounding 2023',             'The Guardian 2015',
             'The Guardian 2018',                'The Vibes 2022',
                  'Treanor 2015',                'Wildhagen 2013',
                   'Wilson 2024']
Length: 25, dtype: str
0 ./interim_results/llm1_geollm_step2_AEMET 2024.csv
1 ./interim_results/llm1_geollm_step2_Koks 2022.csv
2 ./int

In [161]:
df_pred["citation_id"].unique()

<ArrowStringArray>
[                   'AEMET 2024',                     'Koks 2022',
             'Lloyd's List 2024',            'Containerlift 2024',
                      'ABC 2024',              'Karakatsani 2023',
 'European Investment Bank 2025',                   'Wilson 2024',
                'Wildhagen 2013',                      'AFP 2022',
                  'Artemis 2015',                    'Brown 2010',
                      'EFE 2024',                 'Euronews 2024',
                  'Ferlita 2023',                     'Fink 2004',
        'Gilbody Dickerson 2024',                     'Kaur 2025',
                   'Kettle 2020',                   'Khazai 2013',
                'Rozendaal 2021',                'Skoulding 2023']
Length: 22, dtype: str

## TODO check how good "chunk_part" worked out 
Check how well LLM handles the cases where the same CI or LOC occurred multiple times in the chunk, did it still extracted the relevant sentences, despite "e.g.", "US." etc. ?

In [162]:
df_pred.head(1)

,citation_id,chunk_id,infrastructure_type,infrastructure_group,damage,damage_value,location,ci_entity,geo_entity,coord_potential_locations,chunk_text,infrastructure_type_org,damage_org,damage_value_org,locations_org,chunk_part
0,AEMET 2024,20,road infrastructure,NaN,flooded,NaN,NaN,convective train effect,Cuenca,"{'Utiel-Requena': ('39.4697065', '-0.3763353', True), 'Cuenca': ('40.0661031', '-2.1313528', True), 'Valencia': ('39.4189582', '-0.7909637', True), 'Plana de Utiel-Requena': (39.1725, 1.1725, False), 'Buñol Hoya': ('39.4880777', '-1.1001643', True), 'Buñol': ('39.4189582', '-0.7909637', True)}","3rd VICE-PRESIDENCE OF THE GOVERNMENT HNISTERY ECOLOGICAL TRANSITION THE CHALLENGE DEHOGRAPHIC STATE Meteorological Agency Radar Valencia SCM29-I Start date: 29-10-2024 04:00 UTC End date: 29-10-2024 14:00 UTC Type: SCM-LT-PS Transit: Valencia/València, Cuenca Surface effects: torrential and persistent precipitations. Hail and floods Description: Storm line, with at least two supercells embedded, at dawn, very persistent with convective train effect that particularly affected the regions of the Upper Bank, Buñol Hoya and Plana de Utiel-Requena.",road infrastructure,flooded,NaN,"Upper Bank, Buñol Hoya and Plana de Utiel-Requena",NaN


### cleanup entries in prediction set

In [ ]:
## remove entry in df_pred when no "infrastructure_type_org" exists, then it is likely a hallucinated case
df_pred = df_pred.dropna(subset=["infrastructure_type_org"], how="all")


Remove 38 from 471 records which are not CI


#### Remove records which have a FAC (or other) entity in CI_entity ->
-> due that respective LLM response is often faulty (hallucianted)


In [ ]:

## remove entry in df_pred when a "non-CI" record occurs in the "ci_entity" column 
# NOTE this should solves the issue from fix-commit "9fb76a6" - where some ci_entires contained LOCs or non-CI
ci_patterns = pd.read_json("./ner_patterns.jsonl/patterns", lines=True)

# treat removal only on records which have something in "ci_entity" column
tt = df_pred
tt = tt[tt["ci_entity"].notna()]
# get where actual Ci entities are present in "ci_entity" column
tt["ci_entity_grouped"] = None
tt = pp.group_ci_types(tt, col_type="ci_entity", col_grouped="ci_entity_grouped", ci_patterns=ci_patterns)
## get cases of non-CI 
non_ci_records = tt[tt["ci_entity_grouped"].isna()]
# write back - keep only records which are in "ci_entity" either np.nan or a CI type
print(f"Remove {len(non_ci_records)} from {len(df_pred)} records which are not CI")
df_pred = df_pred.drop(index=non_ci_records.index)


#### AS FUNC: Evaluate on same documents that were passed to LLM



In [164]:
print("Use only citations which are in both")


df_valid = df_valid_org[df_valid_org["publication_id"].isin(df_pred.citation_id.unique())]
# df_valid_org[df_valid_org["publication_id"].isin(citation_list)]
print("Valid citations:")
print(df_valid.publication_id.unique())

df_pred = df_pred[df_pred["citation_id"].isin(df_valid.publication_id.unique())]
print("Predicted citations:")
print(df_pred.citation_id.unique())

Use only citations which are in both
Valid citations:
<ArrowStringArray>
[                     'ABC 2024',                    'Brown 2010',
            'Containerlift 2024',                      'EFE 2024',
                 'Euronews 2024', 'European Investment Bank 2025',
                  'Ferlita 2023',        'Gilbody Dickerson 2024',
              'Karakatsani 2023',                     'Kaur 2025',
                   'Khazai 2013',                     'Koks 2022',
                'Rozendaal 2021',                'Wildhagen 2013',
                   'Wilson 2024']
Length: 15, dtype: str
Predicted citations:
<ArrowStringArray>
[                    'Koks 2022',            'Containerlift 2024',
                      'ABC 2024',              'Karakatsani 2023',
 'European Investment Bank 2025',                   'Wilson 2024',
                'Wildhagen 2013',                    'Brown 2010',
                      'EFE 2024',                 'Euronews 2024',
                  'Ferlita

In [165]:
print("Use only citations which are in both")


df_valid = df_valid_org[df_valid_org["publication_id"].isin(df_pred.citation_id.unique())]
# df_valid_org[df_valid_org["publication_id"].isin(citation_list)]
print("Valid citations:")
print(df_valid.publication_id.unique())

df_pred = df_pred[df_pred["citation_id"].isin(df_valid.publication_id.unique())]
print("Predicted citations:")
print(df_pred.citation_id.unique())

Use only citations which are in both
Valid citations:
<ArrowStringArray>
[                     'ABC 2024',                    'Brown 2010',
            'Containerlift 2024',                      'EFE 2024',
                 'Euronews 2024', 'European Investment Bank 2025',
                  'Ferlita 2023',        'Gilbody Dickerson 2024',
              'Karakatsani 2023',                     'Kaur 2025',
                   'Khazai 2013',                     'Koks 2022',
                'Rozendaal 2021',                'Wildhagen 2013',
                   'Wilson 2024']
Length: 15, dtype: str
Predicted citations:
<ArrowStringArray>
[                    'Koks 2022',            'Containerlift 2024',
                      'ABC 2024',              'Karakatsani 2023',
 'European Investment Bank 2025',                   'Wilson 2024',
                'Wildhagen 2013',                    'Brown 2010',
                      'EFE 2024',                 'Euronews 2024',
                  'Ferlita

In [166]:
print(df_valid.info())
print(df_pred.info())

<class 'pandas.DataFrame'>
Index: 121 entries, 3 to 149
Data columns (total 27 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   event_id                118 non-null    str    
 1   event_time              121 non-null    str    
 2   publication_id          121 non-null    str    
 3   sentence_reference      121 non-null    str    
 4   Unnamed: 4              26 non-null     str    
 5   ci1_type                105 non-null    str    
 6   ci_damage_numeric       35 non-null     str    
 7   Unnamed: 7              7 non-null      str    
 8   ci1_damage              97 non-null     str    
 9   ci23_dam_test           11 non-null     str    
 10  ci1_location            100 non-null    str    
 11  ci1_location_spec       37 non-null     str    
 12  ci23_type               31 non-null     str    
 13  ci23_type_number        2 non-null      str    
 14  ci23_location           7 non-null      str    
 15  ci23_

In [167]:
print(df_valid.info())
print(df_pred.info())

<class 'pandas.DataFrame'>
Index: 121 entries, 3 to 149
Data columns (total 27 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   event_id                118 non-null    str    
 1   event_time              121 non-null    str    
 2   publication_id          121 non-null    str    
 3   sentence_reference      121 non-null    str    
 4   Unnamed: 4              26 non-null     str    
 5   ci1_type                105 non-null    str    
 6   ci_damage_numeric       35 non-null     str    
 7   Unnamed: 7              7 non-null      str    
 8   ci1_damage              97 non-null     str    
 9   ci23_dam_test           11 non-null     str    
 10  ci1_location            100 non-null    str    
 11  ci1_location_spec       37 non-null     str    
 12  ci23_type               31 non-null     str    
 13  ci23_type_number        2 non-null      str    
 14  ci23_location           7 non-null      str    
 15  ci23_

In [168]:
# df_pred.loc[df_pred["citation_id"]== "Krausmann 2014"] # Krausmann -> large hallucinations when chunk-text is title or contact info (i.e when not about CI /impacts)

## MV to postprocess.fuc() drop dublicated predictions + upd (encod-utf-8)saving_llm_reuslts in loop (rm fix saving) + pp of NAN strings in LLm response


#### MAKE AS FUNC: group CI into subgroups

In [169]:
ci_patterns = pd.read_json("./ner_patterns.jsonl/patterns", lines=True)


## group Ci types into subgroups,
if not "infrastructure_group" in df_pred.columns or df_pred["infrastructure_group"].isna().any():
    #print("Remove all potential brackets for plural forms in infrastructure types [(s)]")
    #df_pred["infrastructure_type"] = df_pred["infrastructure_type"].str.replace(r"\(s\)", "", regex=True).str.strip()
    df_pred = pp.group_ci_types(df_pred, "infrastructure_type", "infrastructure_group", ci_patterns)
    ## keep only records which are actually about CI (e.g., not theatre, stadion ..)
    df_pred.dropna(subset=["infrastructure_group"], inplace=True)

if not "ci1_group" in df_valid.columns or df_valid["ci1_group"].isna().any():
    df_valid["ci1_group"] = None
    df_valid = pp.group_ci_types(df_valid, "ci1_type", "ci1_group", ci_patterns)
    ## keep only records which are actually about CI (e.g., not theatre, stadion ..)
    df_valid.dropna(subset=["ci1_group"], inplace=True)


print(df_pred.infrastructure_group.isna().sum())  # mostly cases which are not CI (theater, stadion..)
print(df_pred.infrastructure_group.value_counts())
# df_pred.infrastructure_group.unique()


print(df_valid.ci1_group.isna().sum())
print(df_valid.ci1_group.value_counts())
# df_pred.infrastructure_group.unique()


0
infrastructure_group
road_others                     103
rail                             56
it_telecommunication             38
transport_others                 36
water_others                     27
bridges                          14
ports                            14
water_supply                     11
healthcare_hospitals_clinics      6
motorways                         5
waste_others                      5
waterprotection                   5
wastewater                        4
healthcare_others                 3
airports                          3
education_school                  2
electricity_distribution          2
electricity_supply                1
education_kita                    1
gas_distribution                  1
aviation                          1
Name: count, dtype: int64
0
ci1_group
road_others                     34
rail                            12
electricity_others               7
water_supply                     6
airports                         5
motorway

In [170]:

print("Workaround for changing ci_group for drinking water ")

df_valid["ci1_group"] = df_valid["ci1_group"].replace(["drinking_water"], "water_supply")
df_pred["infrastructure_group"] = df_pred["infrastructure_group"].replace(["drinking_water"], "water_supply")
df_pred.infrastructure_group.value_counts()

Workaround for changing ci_group for drinking water 


infrastructure_group
road_others                     103
rail                             56
it_telecommunication             38
transport_others                 36
water_others                     27
bridges                          14
ports                            14
water_supply                     11
healthcare_hospitals_clinics      6
motorways                         5
waste_others                      5
waterprotection                   5
wastewater                        4
healthcare_others                 3
airports                          3
education_school                  2
electricity_distribution          2
electricity_supply                1
education_kita                    1
gas_distribution                  1
aviation                          1
Name: count, dtype: int64

In [171]:
print("Cases of CI which could not be grouped")

print(df_pred[df_pred.infrastructure_group.isna()].shape[0])
print(df_valid[df_valid.ci1_group.isna()].shape[0])

Cases of CI which could not be grouped
0
0


#### MV to postprocess: cases with NANs 

In [172]:

def convert_nan(series: pd.Series) -> pd.Series:
    """ convert representations of "NAN" to np.nan """
    # TODO use regex instead of ["NAN", "NaN", "nan"] by setting all possible representations of nan (e.g. "Nan") to lowercase 
    series = series.replace(["NAN", "NaN", "nan"], np.nan)

    return series


print(df_pred.info())
df_pred["infrastructure_type"] = convert_nan(df_pred["infrastructure_type"])
df_pred["damage"] = convert_nan(df_pred["damage"])
df_pred["location"] = convert_nan(df_pred["location"])

print(df_pred.info(), len(df_pred))



<class 'pandas.DataFrame'>
Index: 338 entries, 27 to 566
Data columns (total 16 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   citation_id                338 non-null    str   
 1   chunk_id                   338 non-null    int64 
 2   infrastructure_type        338 non-null    str   
 3   infrastructure_group       338 non-null    str   
 4   damage                     320 non-null    str   
 5   damage_value               15 non-null     object
 6   location                   336 non-null    str   
 7   ci_entity                  113 non-null    object
 8   geo_entity                 113 non-null    object
 9   coord_potential_locations  338 non-null    str   
 10  chunk_text                 338 non-null    str   
 11  infrastructure_type_org    338 non-null    object
 12  damage_org                 320 non-null    object
 13  damage_value_org           16 non-null     object
 14  locations_org            

#### drop cases in valid and pred where Ci or LOC is empty


In [173]:
print("Removing all records which have erroneous CI or missing LOC entry")

df_pred = df_pred[~df_pred.infrastructure_group.isna()]
df_valid = df_valid[~df_valid.ci1_group.isna()]

df_pred = df_pred[~df_pred.location.isna()]
df_valid = df_valid[~df_valid.ci1_location.isna()]


Removing all records which have erroneous CI or missing LOC entry


In [174]:
## set "affected" to NAN in damage columns (pred, valid)
## TODO check if affected is in general decreasing recall or precision score for "dam" class if yes then set to NAN otherwise keep unchanged


In [175]:
print(len(df_pred))
unique_ci_geo_pairs = df_pred.drop_duplicates()
print("number of duplicates to remove:", len(df_pred) - len(unique_ci_geo_pairs))

df_pred = df_pred.drop_duplicates( )# .reset_index(drop=True, inplace=True)
print(len(df_pred))


333
number of duplicates to remove: 121
212


#### cleanup multiple locs/Cis in one entry (MV FUNC. to pp )

In [176]:
def split_text_into_multiple_rows(df: pd.DataFrame, column: str, split_at = " and ") -> pd.DataFrame:
    """ split text at splitting_point into multiple rows """
    # split CIs and LOCs with "and" into multiple rows
    df[column] = df[column].str.split(split_at)   
    # NOTE: Removes info from CI - drops info if CI is singular o plural (e.g, road and railway infrastrcutre --> "road", "railway infrastructure")
    df = df.explode(column=column)
    df = df.drop_duplicates().reset_index(drop=True)
    return df

# disentangle rows which contain multiple locations or CIs
df_pred = split_text_into_multiple_rows(df_pred, "location",  split_at = " and ")
df_pred = split_text_into_multiple_rows(df_pred, "location",  split_at = " or ")
df_pred = split_text_into_multiple_rows(df_pred, "location",  split_at = ", ")  # Germany, Neterlands, Belgium

df_pred = split_text_into_multiple_rows(df_pred, "infrastructure_type",  split_at = " and ").reset_index(drop=True)

df_valid = split_text_into_multiple_rows(df_valid, "ci1_location",  split_at = " and ")
df_valid = split_text_into_multiple_rows(df_valid, "ci1_location",  split_at = " or ")
df_valid = split_text_into_multiple_rows(df_valid, "ci1_location",  split_at = ", ")
df_valid = split_text_into_multiple_rows(df_valid, "ci1_type",  split_at = " and ").reset_index(drop=True) # make sure that same valid_sentences can occur multiple times, eg. in "Valencia and Sagunto port" (=2 rows)


### remove records which are not about Europe 


In [177]:
df_pred = df_pred[~df_pred["location"].isin(["New York", "New Jersey"])]

### remove all records which are on country-level or "Europe"


In [178]:
# import geonamescache

# def get_countries():
#     geolocs_cache = geonamescache.GeonamesCache()
#     countries = geolocs_cache.get_countries()
#     ci_geo_countries = [*u.gen_dict_extract(countries, 'name')] 
#     # add further country names with abbrev. or "the" , incl. als regions which have the same name as their country (eg. Luxembourg- Provinz in Belgium)
#     ci_geo_countries = ci_geo_countries + ["the Netherlands", "Netherlands", "UK", "US", "U.S.", "USA"]
#     return ci_geo_countries


# # remove all records which are on country-level
# ci_geo_countries = get_countries()

# print(f"Removing {len(df_pred[df_pred['location'].isin(ci_geo_countries)])} records which are on country-level in prediction set")
# print(f"Removing {len(df_valid[df_valid['ci1_location'].isin(ci_geo_countries)])} records which are on country-level in validation set")

# df_pred = df_pred[~df_pred["location"].isin(ci_geo_countries)]
# df_valid = df_valid[~df_valid["ci1_location"].isin(ci_geo_countries)]

## remove all records which mention Europe
df_pred = df_pred[~df_pred["location"].str.contains(r"Europe.*|European.*", case=False, na=False)]
df_valid = df_valid[~df_valid["ci1_location"].str.contains(r"Europe.*|European.*", case=False, na=False)]



# # df_pred[col].apply(lambda x: unidecode(x) if isinstance(x, str) else x)

###### FIXME country-wise removal can be loss of info for certain CI sectors or monetary impacts

In [179]:
# df_pred[df_pred["location"].isin(ci_geo_countries)].drop(["chunk_text","coord_potential_locations"], axis=1)
# df_valid[df_valid["ci1_location"].isin(ci_geo_countries)].drop(["sentence_reference"], axis=1)

## FIXME country-wise removal can be loss of info for certain CI sectors or monetary impacts: 
#        pred: it/telecommunication, waste_*, wastewater; Valid: also gas_supply, electricity infrastructure	

### drop dublicated cases which differ only in Tier 2 or Tier 3 impacts
> e.g. valid ABC 2024: - identical c1_type, ci1_damage, ci1_loc (but diff. ci2_damages -which are not used in this eval) 


In [180]:
print(f"Dropping {df_valid.duplicated().sum()} duplicates in valid data")
df_valid = df_valid.drop_duplicates()

print(f"Dropping {df_pred[['citation_id', 'chunk_id', 'infrastructure_type','damage', 'location', 'chunk_text']].duplicated().sum()} duplicates in pred data")
df_pred = df_pred[df_pred[['citation_id', 'chunk_id', 'infrastructure_type', 'damage', "damage_value", 'location', 'chunk_text']].duplicated() == False]


Dropping 0 duplicates in valid data
Dropping 39 duplicates in pred data


In [181]:
df_pred

,citation_id,chunk_id,infrastructure_type,infrastructure_group,damage,damage_value,location,ci_entity,geo_entity,coord_potential_locations,chunk_text,infrastructure_type_org,damage_org,damage_value_org,locations_org,chunk_part
0,Koks 2022,0,bridge(s),bridges,completely destroyed,NaN,the Netherlands,completely destroyed bridges,the Netherlands,"{'Netherlands': ('51.1638175', '10.4478313', True)}","Nat. Hazards Earth Syst. Sci., 22, 3831-3838, Elco E. Koks 1,2, Kees C. H. van Ginkel 3,1, Margreet J. E. van Marle 3, and Anne Lemnitzer University of Oxford, Oxford, United Kingdom Correspondence: Kees C. H. van Ginkel (kees.vanginkel@deltares.nl) Received: 17 December 2021 - Discussion started: 23 December Revised: 10 August 2022 - Accepted: 18 October 2022 - Published: 29 November Abstract. Germany, Belgium and the Netherlands were hit by extreme precipitation and flooding in July 2021. This brief communication provides an overview of the impacts to large-scale critical infrastructure systems and how recovery has progressed. The results show that Germany and Belgium were particularly affected, with many infrastructure assets severely damaged or completely destroyed. Impacts range from completely destroyed bridges and sewage systems, to severely damaged schools and hospitals.",bridge(s),completely destroyed,NaN,the Netherlands,NaN
1,Koks 2022,0,sewage system(s),wastewater,severely damaged or completely destroyed,NaN,the Netherlands,sewage systems,the Netherlands,"{'Netherlands': ('51.1638175', '10.4478313', True)}","Nat. Hazards Earth Syst. Sci., 22, 3831-3838, Elco E. Koks 1,2, Kees C. H. van Ginkel 3,1, Margreet J. E. van Marle 3, and Anne Lemnitzer University of Oxford, Oxford, United Kingdom Correspondence: Kees C. H. van Ginkel (kees.vanginkel@deltares.nl) Received: 17 December 2021 - Discussion started: 23 December Revised: 10 August 2022 - Accepted: 18 October 2022 - Published: 29 November Abstract. Germany, Belgium and the Netherlands were hit by extreme precipitation and flooding in July 2021. This brief communication provides an overview of the impacts to large-scale critical infrastructure systems and how recovery has progressed. The results show that Germany and Belgium were particularly affected, with many infrastructure assets severely damaged or completely destroyed. Impacts range from completely destroyed bridges and sewage systems, to severely damaged schools and hospitals.",sewage system(s),severely damaged or completely destroyed,NaN,the Netherlands,NaN
2,Koks 2022,0,school(s),education_school,severely damaged,NaN,Germany,NaN,NaN,"{'Netherlands': ('51.1638175', '10.4478313', True)}","Nat. Hazards Earth Syst. Sci., 22, 3831-3838, Elco E. Koks 1,2, Kees C. H. van Ginkel 3,1, Margreet J. E. van Marle 3, and Anne Lemnitzer University of Oxford, Oxford, United Kingdom Correspondence: Kees C. H. van Ginkel (kees.vanginkel@deltares.nl) Received: 17 December 2021 - Discussion started: 23 December Revised: 10 August 2022 - Accepted: 18 October 2022 - Published: 29 November Abstract. Germany, Belgium and the Netherlands were hit by extreme precipitation and flooding in July 2021. This brief communication provides an overview of the impacts to large-scale critical infrastructure systems and how recovery has progressed. The results show that Germany and Belgium were particularly affected, with many infrastructure assets severely damaged or completely destroyed. Impacts range from completely destroyed bridges and sewage systems, to severely damaged schools and hospitals.",school(s),severely damaged,NaN,Germany and Belgium,NaN
3,Koks 2022,0,school(s),education_school,severely damaged,NaN,Belgium,NaN,NaN,"{'Netherlands': ('51.1638175', '10.4478313', True)}","Nat. Hazards Earth Syst. Sci., 22, 3831-3838, Elco E. Koks 1,2, Kees C. H. van Ginkel 3,1, Margreet J. E. van Marle 3, and Anne Lemnitzer University of Oxford, Oxford, United Kingdom Correspondence: Kees C. H. van Ginkel (kees.vanginkel@deltares.nl) Received: 17 December 20

#### AS FUNC: clean-up locations

In [182]:

print(df_pred.location.unique())
print(df_valid.ci1_location.unique())

## TODO clean-up locs --> MV this postprocessing to LLm extraction before passing to LLm Step2
# "( "  eg "Sinzig (in North Rhine-Westphalia)""
df_pred["location"] = df_pred["location"].str.split(r"\(", regex=True).str[0].str.strip()
df_valid["ci1_location"] = df_valid["ci1_location"].str.split(r"\(", regex=True).str[0].str.strip()

# rm text after comma, e.g. "Ahr valley, Germany" --> "Ahr valley"
df_pred["location"] = df_pred["location"].str.split(", ").str[0].str.strip()  
df_valid["ci1_location"] = df_valid["ci1_location"].str.split(", ").str[0].str.strip()  
# remove all remaining brackets and commas
df_pred["location"] = df_pred["location"].replace(r"[\(\),]", "", regex=True)
df_valid["ci1_location"] = df_valid["ci1_location"].replace(r"[\(\),]", "", regex=True)
## remove "the"
df_pred["location"] = df_pred["location"].replace("the ", "")
df_valid["ci1_location"] = df_valid["ci1_location"].replace("the ", "")

## handling loc with "railway tracks between "
df_pred["location"] = df_pred["location"].str.split("between").str[-1].str.strip()
df_valid["ci1_location"] = df_valid["ci1_location"].str.split("between").str[-1].str.strip()
## handling loc with "railway tracks in"  , set this after cleaning up "( " and ", " as it otherwise would take the later location
df_pred["location"] = df_pred["location"].str.split("in ").str[-1].str.strip()
df_valid["ci1_location"] = df_valid["ci1_location"].str.split("in ").str[-1].str.strip()
#  try to remove "the" already before passing to GeoLLM in extraction-WF
df_pred["location"] = df_pred["location"].replace(r"the ", " ", regex=False).str.strip()   
df_valid["ci1_location"] = df_valid["ci1_location"].replace(r"the ", " ", regex=False).str.strip()
## handling loc with "passing "
df_pred["location"] = df_pred["location"].replace(r"passing ", " ", regex=False).str.strip()   
df_valid["ci1_location"] = df_valid["ci1_location"].replace(r"passing ", " ", regex=False).str.strip()


print("After cleanup: drop records which are not about CI anymore")
ci_patterns = pd.read_json("./ner_patterns.jsonl/patterns", lines=True)

df_pred = pp.group_ci_types(df_pred, "infrastructure_type", "infrastructure_group", ci_patterns)
## keep only records which are actually about CI (e.g., not remainings from cleanup)
df_pred.dropna(subset=["infrastructure_group"], inplace=True)


print(df_pred.location.unique())
print(df_valid.ci1_location.unique())

<ArrowStringArray>
[                            'the Netherlands',
                                     'Germany',
                                     'Belgium',
                        'Eiffel National Park',
                               'city of Trier',
 'Vesdre River valley (districts of Pepinster',
                                     'Ensival',
                                   'Verviers)',
                 'Meuse River valley (Maaseik',
                                      'Liége)',
 ...
                        'Saale-Holzland-Kreis',
                           'Wartburg district',
                                  'Regensburg',
                                   'Magdeburg',
                        'Walloon rail network',
                              'between Bunnik',
                                  'Veenendaal',
                             'between Utrecht',
                              'Ede-Wageningen',
                                      'Rhenen']
Length: 131, dty

### locations_2_coordinates() matching, MAKE AS FUNC

In [183]:
def get_bbox(points):
    x_coordinates, y_coordinates = zip(*points)
    return [(min(x_coordinates), min(y_coordinates)), (max(x_coordinates), max(y_coordinates))]

# TODO mv to utils.py or geolocalization.py

In [184]:
# extract dict from geollm-string
df_pred["coord_potential_locations"] = df_pred["coord_potential_locations"].apply(lambda x: eval(str(x))) 


df_pred["coords"] = None
df_pred["coords_coarse"] = None

# reindex to avoid issues in iloc
df_pred.reset_index(drop=True, inplace=True)

## get most likely coordinates for locations
counter_errors = 0
for entry in range(len(df_pred)):

    loc = df_pred["location"].iloc[entry]
    loc_potential_coords = df_pred["coord_potential_locations"].iloc[entry]
    try:
        # go to next entity when it is NAN
        if loc == np.nan or loc == "nan" or loc == "NAN" or loc == "NaN" or loc == "":
            continue
        # write coordinates to df
        df_pred.at[entry,  "coords"] = loc_potential_coords[loc][0:2]

    except Exception as e:
        try:
            for coords in loc_potential_coords.keys(): 

                ## WORKAROUND handling "Ahr river valley" <-> "Ahrtal"
                if coords == "Ahrtal":  # geollama response
                    df_pred.at[entry, "location"] = df_pred.at[entry,"location"].replace("Ahr River valley", "Ahrtal")
                    loc = df_pred["location"].iloc[entry]

                smlrty = fuzz.partial_ratio(loc, coords)
                if smlrty > 90: 
                    print("\nUsing partial ratio for matching:", loc, "<->", coords, f"{smlrty}")
                    # NOTE: handles "Malaga airport", "the Ahr valley", "Erft region"
                    # FIXME needs imrovement in the future, to get more concrete spat. info (if it is region, a river etc.)
                    df_pred.at[entry,  "coords"] = loc_potential_coords[coords][0:2]
                else: pass
       
        except Exception as e:
            print(f"\nEntry {entry}: {df_pred['location'].iloc[entry]}")
            print(f"No geolocalization possible for row {entry}: {e} \n{df_pred['coord_potential_locations'].iloc[entry]}")
            # e.g. "flood region", "A76 in both directions"
            print("Creating BBox of potential location based on locations mentioned in respective chunk text")
            coords_list = [[float(v[0]), float(v[1])] for v in loc_potential_coords.values()]
            df_pred.at[entry,  "coords_coarse"] =  get_bbox(coords_list)
            counter_errors = counter_errors + 1


print(f"{counter_errors} Cases where only coarse loc could be extracted ( based on loc in entire chunk):")
## drop cases where no geolocalization could be done 


# # TODO when loc= "A76 in both directions" --> make new column with "eigenname" new column "potentially_location_in" with list of geollm returns and bbox based on these geollm_locs
# # TODO measure location new based on centroid of loc_red or centroid of "potentially_in"


Using partial ratio for matching: the Netherlands <-> Netherlands 100

Using partial ratio for matching: the Netherlands <-> Netherlands 100

Using partial ratio for matching: city of Trier <-> Trier 100

Using partial ratio for matching: Vesdre River valley <-> Vesdre River 100

Using partial ratio for matching: Gete River valley <-> Gete River 100

Using partial ratio for matching: southeast Brussels <-> Brussels 100

Using partial ratio for matching: the Ahr valley <-> Ahr 100

Using partial ratio for matching: the Ahr valley <-> Ahr 100

Using partial ratio for matching: the Ahr valley <-> Ahr 100

Using partial ratio for matching: the Ahr valley <-> Ahr 100

Using partial ratio for matching: the Ahr valley <-> Ahr 100

Using partial ratio for matching: the Ahr valley <-> Ahr 100

Using partial ratio for matching: the Ahr valley <-> Ahr 100

Using partial ratio for matching: the Ahr valley <-> Ahr 100

Using partial ratio for matching: the Ahr valley <-> Ahr 100

Using partial rat

In [185]:
len(df_pred)

209

In [186]:
df_pred[[
    "citation_id","chunk_id", "infrastructure_type", "infrastructure_group", "damage", "damage_value", "location", "ci_entity",	"geo_entity", "coord_potential_locations", "infrastructure_type_org",	"damage_org","damage_value_org","locations_org","coords","coords_coarse"
    ]][4:50]  
# check for duplicates


,citation_id,chunk_id,infrastructure_type,infrastructure_group,damage,damage_value,location,ci_entity,geo_entity,coord_potential_locations,infrastructure_type_org,damage_org,damage_value_org,locations_org,coords,coords_coarse
4,Koks 2022,0,hospital(s),healthcare_hospitals_clinics,severely damaged,NaN,Germany,NaN,NaN,"{'Netherlands': ('51.1638175', '10.4478313', True)}",hospital(s),severely damaged,NaN,Germany and Belgium,None,None
5,Koks 2022,0,hospital(s),healthcare_hospitals_clinics,severely damaged,NaN,Belgium,NaN,NaN,"{'Netherlands': ('51.1638175', '10.4478313', True)}",hospital(s),severely damaged,NaN,Germany and Belgium,None,None
6,Koks 2022,2,road infrastructure,road_others,flooded,NaN,Eiffel National Park,NaN,NaN,"{'Belgium': ('50.6402809', '4.6667145', True), 'Dutch': ('51.2015196', '5.9046302', True), 'Trier': ('49.7596208', '6.6441878', True), 'Herk-de-Stad': ('50.9222911', '5.1884302', True), 'Halen': ('50.9382257', '5.1073800', True), 'Verviers': ('50.5932400', '5.8678280', True), 'Vesdre River': (50.3262, 5.0162, False), 'Rheinland-Pfalz': ('49.9531599', '7.3106460', True), 'Liége': ('50.4708135', '5.7735658', True), 'Wavre': ('50.7231225', '4.6152531', True), 'Brussels': ('50.8467372', '4.3524930', True), 'Gete River': ('50.9529614', '5.1183724', True), 'Limburg': (50.6145118, 5.9406195, True), 'Ensival': ('50.5785616', '5.8479666', True), 'Maaseik': ('51.1480712', '5.5549907', True), 'Eiffel National Park': (49.7, 5.5, False), 'Ahrtal': ('50.3900652', '6.7335837', True)}",road infrastructure,flooded,NaN,Eiffel National Park,"(49.7, 5.5)",None
7,Koks 2022,2,road infrastructure,road_others,flooded,NaN,city of Trier,NaN,NaN,"{'Belgium': ('50.6402809', '4.6667145', True), 'Dutch': ('51.2015196', '5.9046302', True), 'Trier': ('49.7596208', '6.6441878', True), 'Herk-de-Stad': ('50.9222911', '5.1884302', True), 'Halen': ('50.9382257', '5.1073800', True), 'Verviers': ('50.5932400', '5.8678280', True), 'Vesdre River': (50.3262, 5.0162, False), 'Rheinland-Pfalz': ('49.9531599', '7.3106460', True), 'Liége': ('50.4708135', '5.7735658', True), 'Wavre': ('50.7231225', '4.6152531', True), 'Brussels': ('50.8467372', '4.3524930', True), 'Gete River': ('50.9529614', '5.1183724', True), 'Limburg': (50.6145118, 5.9406195, True), 'Ensival': ('50.5785616', '5.8479666', True), 'Maaseik': ('51.1480712', '5.5549907', True), 'Eiffel National Park': (49.7, 5.5, False), 'Ahrtal': ('50.3900652', '6.7335837', True)}",road infrastructure,flooded,NaN,city of Trier,"(49.7596208, 6.6441878)",None
8,Koks 2022,2,road infrastructure,road_others,flooded,NaN,Vesdre River valley,NaN,NaN,"{'Belgium': ('50.6402809', '4.6667145', True), 'Dutch': ('51.2015196', '5.9046302', True), 'Trier': ('49.7596208', '6.6441878', True), 'Herk-de-Stad': ('50.9222911', '5.1884302', True), 'Halen': ('50.9382257', '5.1073800', True), 'Verviers': ('50.5932400', '5.8678280', True), 'Vesdre River': (50.3262, 5.0162, False), 'Rheinland-Pfalz': ('49.9531599', '7.3106460', True), 'Liége': ('50.4708135', '5.7735658', True), 'Wavre': ('50.7231225', '4.6152531', True), 'Brussels': ('50.8467372', '4.3524930', True), 'Gete River': ('50.9529614', '5.1183724', True), 'Limburg': (50.6145118, 5.9406195, True), 'Ensival': ('50.5785616', '5.8479666', True), 'Maaseik': ('51.1480712', '5.5549907', True), 'Eiffel National Park': (49.7, 5.5, False), 'Ahrtal': ('50.3900652', '6.7335837', True)}",road infrastructure,flooded,NaN,"Vesdre River valley (districts of Pepinster, Ensival and Verviers)","(50.3262, 5.0162)",None
9,Koks 2022,2,road infrastructure,road_others,flooded,NaN,Ensival,NaN,NaN,"{'Belgium': ('50.6402809', '4.6667145', True), 'Dutch': ('51.2015196', '5.9046302', True), 'Trier': ('49.7596208', '6.6441878', True), 'Herk-de-Stad': ('50.9222911', '5.1884302', True), 'Halen': ('50.9382257', '5.1073800', True), 'Verviers': ('50.5932400', '5.8678280', True), 'Vesdre River': (50.3262, 5.0162, False), 'Rheinland-Pfalz': ('49.9531599', '7.3106460', True), 'Liége': ('50.4708135', '5.7735658

##### Loc_2_coords mathcing for validation set

In [187]:
# extract dict from geollm-string
df_pred["coord_potential_locations"] = df_pred["coord_potential_locations"].apply(lambda x: eval(str(x))) 


df_pred["coords"] = None
df_pred["coords_coarse"] = None

# reindex to avoid issues in iloc
df_pred.reset_index(drop=True, inplace=True)

## get most likely coordinates for locations
counter_errors = 0
for entry in range(len(df_pred)):

    loc = df_pred["location"].iloc[entry]
    loc_potential_coords = df_pred["coord_potential_locations"].iloc[entry]
    try:
        # go to next entity when it is NAN
        if loc == np.nan or loc == "nan" or loc == "NAN" or loc == "NaN" or loc == "":
            continue
        # write coordinates to df
        df_pred.at[entry,  "coords"] = loc_potential_coords[loc][0:2]

    except Exception as e:
        try:
            for coords in loc_potential_coords.keys(): 

                ## WORKAROUND handling "Ahr river valley" <-> "Ahrtal"
                if coords == "Ahrtal":  # geollama response
                    df_pred.at[entry, "location"] = df_pred.at[entry,"location"].replace("Ahr River valley", "Ahrtal")
                    loc = df_pred["location"].iloc[entry]

                smlrty = fuzz.partial_ratio(loc, coords)
                if smlrty > 90: 
                    print("\nUsing partial ratio for matching:", loc, "<->", coords, f"{smlrty}")
                    # NOTE: handles "Malaga airport", "the Ahr valley", "Erft region"
                    # FIXME needs imrovement in the future, to get more concrete spat. info (if it is region, a river etc.)
                    df_pred.at[entry,  "coords"] = loc_potential_coords[coords][0:2]
                else: pass
       
        except Exception as e:
            print(f"\nEntry {entry}: {df_pred['location'].iloc[entry]}")
            print(f"No geolocalization possible for row {entry}: {e} \n{df_pred['coord_potential_locations'].iloc[entry]}")
            # e.g. "flood region", "A76 in both directions"
            print("Creating BBox of potential location based on locations mentioned in respective chunk text")
            coords_list = [[float(v[0]), float(v[1])] for v in loc_potential_coords.values()]
            df_pred.at[entry,  "coords_coarse"] =  get_bbox(coords_list)
            counter_errors = counter_errors + 1


print(f"{counter_errors} Cases where only coarse loc could be extracted ( based on loc in entire chunk):")
## drop cases where no geolocalization could be done 


# # TODO when loc= "A76 in both directions" --> make new column with "eigenname" new column "potentially_location_in" with list of geollm returns and bbox based on these geollm_locs
# # TODO measure location new based on centroid of loc_red or centroid of "potentially_in"


Using partial ratio for matching: the Netherlands <-> Netherlands 100

Using partial ratio for matching: the Netherlands <-> Netherlands 100

Using partial ratio for matching: city of Trier <-> Trier 100

Using partial ratio for matching: Vesdre River valley <-> Vesdre River 100

Using partial ratio for matching: Gete River valley <-> Gete River 100

Using partial ratio for matching: southeast Brussels <-> Brussels 100

Using partial ratio for matching: the Ahr valley <-> Ahr 100

Using partial ratio for matching: the Ahr valley <-> Ahr 100

Using partial ratio for matching: the Ahr valley <-> Ahr 100

Using partial ratio for matching: the Ahr valley <-> Ahr 100

Using partial ratio for matching: the Ahr valley <-> Ahr 100

Using partial ratio for matching: the Ahr valley <-> Ahr 100

Using partial ratio for matching: the Ahr valley <-> Ahr 100

Using partial ratio for matching: the Ahr valley <-> Ahr 100

Using partial ratio for matching: the Ahr valley <-> Ahr 100

Using partial rat

#### Load spaCy language model


In [188]:
## load english model with contextual vectors included


## RELOAD spacy pipeline
nlp = spacy.load("./spacy_model_pipeline")


#### Select records which have text references

In [189]:
df_valid.info()

<class 'pandas.DataFrame'>
RangeIndex: 94 entries, 0 to 93
Data columns (total 28 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   event_id                93 non-null     str    
 1   event_time              94 non-null     str    
 2   publication_id          94 non-null     str    
 3   sentence_reference      94 non-null     str    
 4   Unnamed: 4              16 non-null     str    
 5   ci1_type                94 non-null     str    
 6   ci_damage_numeric       32 non-null     str    
 7   Unnamed: 7              5 non-null      str    
 8   ci1_damage              87 non-null     str    
 9   ci23_dam_test           10 non-null     str    
 10  ci1_location            94 non-null     object 
 11  ci1_location_spec       36 non-null     str    
 12  ci23_type               30 non-null     str    
 13  ci23_type_number        1 non-null      str    
 14  ci23_location           6 non-null      str    
 15  ci

In [190]:
print(len(df_pred), len(df_valid))
df_pred = df_pred[~df_pred["chunk_text"].isna()].reset_index(drop=True)
df_valid = df_valid[~df_valid["sentence_reference"].isna()].reset_index(drop=True)
print(len(df_pred), len(df_valid))


209 94
209 94


#### Translation of validation sentences

In [191]:

for entry in df_valid.itertuples():
    
    src_language = langdetect.detect(str(entry.sentence_reference))
    
    if src_language != "en":
        supported_languages = ["fr", "de", "es", "it", "itc", "nl"]
        if src_language not in supported_languages:
            print(f"Unsupported source language: {src_language}. Continue with original version of the sentence in validation set ")
            continue 

        print(f"\n ######## -------- Translating {entry.publication_id}: {src_language} --> en -------- ######## \n")

        # # clean up before applying translator
        # gc.collect()
        # torch.cuda.empty_cache()  # mainly after training needed, small effect when LLM applied only for inference
        # torch.no_grad()
        
        # overwrite original sentence(s) with translated versions
        translated_sentence = tm.translate_2_english(src_language, str(entry.sentence_reference))
        df_valid.loc[df_valid.index[df_valid["sentence_reference"] == entry.sentence_reference], "sentence_reference"] = translated_sentence


In [192]:
df_valid.info()

<class 'pandas.DataFrame'>
RangeIndex: 94 entries, 0 to 93
Data columns (total 28 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   event_id                93 non-null     str    
 1   event_time              94 non-null     str    
 2   publication_id          94 non-null     str    
 3   sentence_reference      94 non-null     str    
 4   Unnamed: 4              16 non-null     str    
 5   ci1_type                94 non-null     str    
 6   ci_damage_numeric       32 non-null     str    
 7   Unnamed: 7              5 non-null      str    
 8   ci1_damage              87 non-null     str    
 9   ci23_dam_test           10 non-null     str    
 10  ci1_location            94 non-null     object 
 11  ci1_location_spec       36 non-null     str    
 12  ci23_type               30 non-null     str    
 13  ci23_type_number        1 non-null      str    
 14  ci23_location           6 non-null      str    
 15  ci

In [193]:
# # unicode to ascii representation
# print("Apply unicode on CI and damages, but not on Locations (with ä, ü and other special chars) as it removes them potentially from the DFs" )
# try:
#     for col in ["infrastructure_group", "infrastructure_type", "damage", "location", "ci_entity", "geo_entity"]:
#         df_pred[col] = df_pred[col].apply(lambda x: unidecode(x) if isinstance(x, str) else x) # handle potential np.nan
# except KeyError as e:
#     for col in ["infrastructure_group", "infrastructure_type", "damage", "location"]:
#         df_pred[col] = df_pred[col].apply(lambda x: unidecode(x) if isinstance(x, str) else x) # handle potential np.nan

# for col in ["ci1_group", "ci1_type", "ci1_damage", "ci1_location"]:
#     df_valid[col] = df_valid[col].apply(lambda x: unidecode(x) if isinstance(x, str) else x) # handle potential np.nan


#### add unique identifiers
helps in calculating FPs and FNs


In [194]:
# df_pred[["citation_id",	"chunk_id",	"infrastructure_type",	"infrastructure_group",	"damage",	"location"	]]

df_pred.loc[:,"id_pred"] = df_pred.reset_index().index
df_valid.loc[:,"id_valid"] = df_valid.reset_index().index

#### Merge prediction entries with potential validation entries (nth:1 pairs)

In [195]:
import re


print("Match chunk text of each prediction entry with related validation entries (nth:1 pairs)")
print("Align texts from valid and pred set by removing potential whitespaces")
# NOTE Solves issue: of having doubled whitespace or whitespaces due to linebreaks. eg. "Bad Münstereifel" where valid senence differed to pred_chunk due to "- " (instead of "-") in fresh-water

df_pred_valid_all = pd.DataFrame()
threshold = 65  # keep low due to differences in the tranlsation and when linebreaks where used

# find for each prediction entry all validation entries for respective chunk 
# these validation entries are candidates from which the most similar one to the pred. entry is taken to calc. model performance 
# including also entries where pred_info or valid_info is missing (e.g FNs, FPs)
for _, pred_entry in df_pred.iterrows():
    for _, valid_entry in df_valid.iterrows():  # all validation entries of all docs

        if valid_entry.sentence_reference is np.nan:
            continue

        valid_entry.sentence_reference = re.sub(r"([^\s-])\n([^\s-])", r"\1 \2", valid_entry.sentence_reference) # replace linebreak symbols when they occur just once, with whitespace (two linebreaks - probably new subsection)
        valid_entry.sentence_reference = valid_entry.sentence_reference.replace("/\n{2,}/g", "\n")  # remove linebreaks only when they occurred just once, but not for multiple linebreaks (e.g. before subsection)
        valid_entry.sentence_reference = re.sub(r"\s+", " ", valid_entry.sentence_reference)  # replace >1 whitespaces with single whitespace
        valid_entry.sentence_reference = re.sub(r"([^\s-])- ([^\s-])", r"\1-\2", valid_entry.sentence_reference)  # remove hypens in the middle of lines
        
        pred_entry.chunk_text = re.sub(r"([^\s-])\n([^\s-])", r"\1 \2", pred_entry.chunk_text) # replace linebreak symbols when they occur just once, with whitespace (two linebreaks - probably new subsection)
        pred_entry.chunk_text = pred_entry.chunk_text.replace("/\n{2,}/g", "\n")  # remove linebreaks only when they occurred just once, but not for multiple linebreaks (e.g. before subsection)
        pred_entry.chunk_text = re.sub(r"\s+", " ", pred_entry.chunk_text)  # replace >1 whitespaces with single whitespace
        pred_entry.chunk_text = re.sub(r"([^\s-])- ([^\s-])", r"\1-\2", pred_entry.chunk_text)  # remove hypens in the middle of lines

        # Calculate match score by accounting for partial string matches. 
        # In detail, it calculates the similarity ratio using the shortest string (length n, here: "sentence_reference") against all n-length substrings of the larger string and returns the highest score 
        score = fuzz.partial_ratio(valid_entry['sentence_reference'].replace(" ", ""), pred_entry['chunk_text'].replace(" ", ""))



        if score >= threshold:
            entry_pred_valid = {
                "citation_id": pred_entry["citation_id"],
                "ci_pred": pred_entry["infrastructure_type"],
                "ci_group_pred": pred_entry["infrastructure_group"],
                "damage_pred": pred_entry["damage"],
                "location_pred": pred_entry["location"],
                "coords_pred": pred_entry["coords"],
                "chunk_id_pred": pred_entry["chunk_id"],
                "chunk_text_pred": pred_entry["chunk_text"],
                "ci_valid": valid_entry["ci1_type"],
                "ci_group_valid": valid_entry["ci1_group"],
                "damage_valid": valid_entry["ci1_damage"],
                "location_valid": valid_entry["ci1_location"],
                # "coords_valid": valid_entry["coords"],
                "sentence_text_valid": valid_entry["sentence_reference"],
                "text_similarity": score,
                "id_pred": pred_entry["id_pred"],
                "id_valid": valid_entry["id_valid"]
            }
            df_pred_valid_all = pd.concat([df_pred_valid_all, pd.DataFrame([entry_pred_valid])], ignore_index=True)  # n:1 relationship DF
        

# 85 threshold - 778 entries
# 75 threshold - 778 entries
# 75 threshold + CIsubgrou - 513 entries




Match chunk text of each prediction entry with related validation entries (nth:1 pairs)
Align texts from valid and pred set by removing potential whitespaces


In [196]:
df_pred_valid_all.info() # 143 -190 entries

## --> FPs are more common compared to FNs, especially for predicting locations, 
# as it is easier to get a prep-valid match when pred.info is actually missing due to larger chunk-text (pred set) compared to sentence-text (valid set)


<class 'pandas.DataFrame'>
RangeIndex: 203 entries, 0 to 202
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   citation_id          203 non-null    str   
 1   ci_pred              203 non-null    str   
 2   ci_group_pred        203 non-null    str   
 3   damage_pred          201 non-null    object
 4   location_pred        203 non-null    str   
 5   coords_pred          168 non-null    object
 6   chunk_id_pred        203 non-null    int64 
 7   chunk_text_pred      203 non-null    str   
 8   ci_valid             203 non-null    str   
 9   ci_group_valid       203 non-null    str   
 10  damage_valid         178 non-null    object
 11  location_valid       203 non-null    str   
 12  sentence_text_valid  203 non-null    str   
 13  text_similarity      203 non-null    int64 
 14  id_pred              203 non-null    int64 
 15  id_valid             203 non-null    int64 
dtypes: int64(4), object

In [197]:
## entries with lowest similarity
df_pred_valid_all.text_similarity.describe() # 413  (no regex, step2) , 284 (no c regex, step1)
# df_pred_valid_all.iloc[df_pred_valid_all.text_similarity.sort_values(ascending=True).index] [["sentence_text_valid", "chunk_text_pred","text_similarity"]]

count    203.000000
mean      98.689655
std        2.652710
min       83.000000
25%       99.000000
50%      100.000000
75%      100.000000
max      100.000000
Name: text_similarity, dtype: float64

## Document-wise evaluation

Measures simply if the predicted CI-LOC case also occurs in the validation set\
It does not measure the frequency - just if the CI-LOC case exists in the validation set. In this way, the approach is similar to a spatial evaluation which also just captures the occurrence and location, not the frequency with which the impact (ie CI-LOC case) was reported in the document 


In [198]:
## TEST rm records where no geolocalization was possible (incl. coarse coords)
print(f"Removing {len(df_pred[df_pred['coords'].isna()])} from {len(df_pred)} records where no geolocalization was possible (incl. coarse coords)")

df_pred_geolocalized = df_pred[~df_pred["coords"].isna()]


Removing 58 from 209 records where no geolocalization was possible (incl. coarse coords)


In [199]:
for publication in df_valid.publication_id.unique():

    print(f"\n\nDocument: {publication}")
    docs_valid = df_valid[df_valid["publication_id"] == publication]
    docs_pred = df_pred_geolocalized[df_pred_geolocalized["citation_id"] == publication]

    # get all valid and predicted CI-LOC pairs for each publication
    docs_valid_pairs = docs_valid[["ci1_group", "ci1_location"]].drop_duplicates().values.tolist()
    docs_pred_pairs = docs_pred[["infrastructure_group", "location"]].drop_duplicates().values.tolist()
    print(docs_valid_pairs)
    print(docs_pred_pairs)

    # calc TPs, FPs, FNs
    tps = len([t for t in docs_pred_pairs if t in docs_valid_pairs]) 
    fps = len([t for t in docs_pred_pairs if t not in docs_valid_pairs]) 
    fns = len([t for t in docs_valid_pairs if t not in docs_pred_pairs]) # geo-llama.trsting_on_news2024.ipynb
    print(f"TPs: {tps}, FPs: {fps}, FNs: {fns}")
    print(f"  FPs: { [t for t in docs_pred_pairs if t not in docs_valid_pairs]}")



Document: ABC 2024


[['airports', 'Malaga area']]
[['transport_others', 'Fuengirola'], ['transport_others', 'Benalmádena'], ['transport_others', 'Mijas'], ['transport_others', 'Torrox'], ['transport_others', 'Ronda'], ['transport_others', 'Casabermeja'], ['transport_others', 'Cártama'], ['transport_others', 'Campillos'], ['airports', 'Malaga'], ['water_others', 'Málaga'], ['water_others', 'Paseo de la Farola'], ['water_others', 'Pujerra'], ['water_others', 'Sierra de Mijas'], ['water_others', 'Guadalhorce River'], ['water_others', 'Torre del Mar'], ['road_others', 'Velázquez Avenue'], ['road_others', 'Pasillo del Matadero Puente del Carmen'], ['road_others', 'Pasillo Santa Isabel - Puente de la Aurora'], ['road_others', 'Avenida Lope de Vega - Atabal'], ['road_others', 'Colonia Santa Inés'], ['road_others', 'Juan XXIII Avenue'], ['road_others', 'Plaza Manuel Azaña'], ['road_others', 'Victoria']]
TPs: 0, FPs: 23, FNs: 1
  FPs: [['transport_others', 'Fuengirola'], ['transport_others', 'Benalmádena'], ['tran

In [200]:
print(len(df_valid), len(df_pred), len(df_pred_geolocalized))
df_pred = df_pred_geolocalized


94 209 151


## Calc similarities 
* TPs (for all cases where text info in pred and valid set exists)
* FNs  (model missed actual cases)
* FPs  (model hallucinated cases)

In [201]:
list_entity_valid = ["ci_group_valid", "damage_valid", "location_valid"]
list_entity_pred = ["ci_group_pred", "damage_pred", "location_pred"]



#  Set similarity threshold (self-defined) when CI case is valid or not FN/FP
cos_smlrty_thresh = 0.7
norm_pr_smlrty_thresh = 0.7


print(" --- For each unique valid case (unique combi: [ci_valid, damage_valid, location_valid, sentence_text]) calculate similarity ---")
print("Using 100% match for CI types based on subgroups")
print("Using cosine similarity threshold for damages", cos_smlrty_thresh)
print("Using normalized partial ratio similarity threshold for locations", norm_pr_smlrty_thresh)

## AIM of evaluation loop below: 
# remove all cases in df_pred_valid_all where pred_entities were wrongly assigned to a valid_entity
## ie keep only pre-valid pairs with highest similarity per unique valid case



df_eval_records = pd.DataFrame()


# For each impact case (rows) 
for record_no, impact_record in df_pred_valid_all.iterrows():

    print(f"Record: {record_no } / {len(df_pred_valid_all)}")

    # init dict to store results for each records (row=)
    df_eval = {
        "citation": impact_record.citation_id,
        "chunk_text_pred": impact_record.chunk_text_pred,
        "sentence_text_valid": impact_record.sentence_text_valid,
        "id_pred": impact_record.id_pred,
        "id_valid": impact_record.id_valid
    }
    
    # iterate over the three entity classes (ci, damage, location) to assess LLM performance
    for entity_valid, entity_pred in zip(list_entity_valid, list_entity_pred):


        # Calculate similarities for entries in column pair: entity_pred - entity_valid

        ## calc similarity when both valid_info exist (not NAN) 
        # NOTE df_pred model can put out NAN when case exists but it couldnt find suitable value (e.g. damage_pred="NaN", damage_valid="polluted")
        if impact_record[entity_valid] is not np.nan:
        # if impact_record[entity_pred] and impact_record[entity_valid] is not np.nan:
            pred_impact = impact_record[entity_pred]
            valid_impact = impact_record[entity_valid]

            if entity_pred == "ci_group_pred": # for CI group, only partial ratio similarity is calculated as it is more important to get the correct group than the exact match (e.g. "port infrastructure" <-> "port")

                # embedded_list = u.vector_calculation(pred_impact, valid_impact)
                # similarity_score_cos = u.cosine_similarity(embedded_list[0], embedded_list[1])
                # similarity_score_pr = np.nan
                
                # similarity on idential match 
                if pred_impact == valid_impact:
                    ci_smlrty = 1
                else:
                    ci_smlrty = 0

                # store result for ci entity 
                df_eval["ci_pred"] = pred_impact  # CI subgroup
                df_eval["ci_valid"] = valid_impact # CI subgroup
                df_eval["ci_smlrty"] = ci_smlrty

            if entity_pred == "damage_pred": 

                if isinstance(pred_impact, str) & isinstance(valid_impact, str): # check that pred or valid are not NAN
                    # contextual vectors (transformer-based)
                    embedded_list = u.vector_calculation(pred_impact, valid_impact)
                    # calculate cosine similarity for each pred-valid damage pair
                    similarity_score_cos = u.cosine_similarity(embedded_list[0], embedded_list[1])  # 0-1 value, the higher the more similar
                
                # if nan -> then it is either FP or FN
                elif isinstance(pred_impact, float) | isinstance(np.nan, float):
                    similarity_score_cos = 0.0     

                # store result for DAM and LOC entity 
                df_eval["dam_pred"] = pred_impact  
                df_eval["dam_valid"] = valid_impact 
                df_eval["dam_smlrty"] = similarity_score_cos


            if entity_pred == "location_pred": 
                ## Cosine similarity calc.
                if isinstance(pred_impact, str):
                    # contextual vectors (transformer-based)
                    embedded_list = u.vector_calculation(pred_impact, valid_impact)
                    # calculate cosine similarity for each pred-valid pair
                    similarity_score_cos = u.cosine_similarity(embedded_list[0], embedded_list[1])  # 0-1 value, the higher the more similar
                elif np.isnan(pred_impact):
                    similarity_score_cos = 0.0   # NOTE it is FNs
                ## Partial ratio similarity calc. (especially for locations and CI-type  "port infrastructure" <-> "port")
                similarity_score_pr = fuzz.partial_ratio(pred_impact, valid_impact)  

                # store result for DAM and LOC entity 
                df_eval["loc_pred"] = pred_impact  
                df_eval["loc_valid"] = valid_impact 
                df_eval["loc_smlrty"] = similarity_score_cos
                df_eval["loc_smlrty_norm_pr"] = similarity_score_pr / 100

    # collect all single records (row) with similarity scores
    df_eval_records = pd.concat([df_eval_records, pd.DataFrame([df_eval])], ignore_index=True)


## NOTE Description: How 1:1 pairs for pred-valid are extracted f
## 1. group by single records from df_valid (via id_valid indices), 
##    Column "id_valid": index represents single records from df_valid (when validation_sentence contains 2 cases: -> id-valid:0, id_valid:1,  sentence w 1 case: id-valid:2) 
## 2. then collect from each group the one with highest similarity to predictions
##    --> binary "mask" indicates where we have matches -e.g. correct predictions (true: TP, false: FN or FP)  is our match (1:1 pred-valid pair) - from which TPs can be calculated

## 1. + 2.
# select for each single valid record (ie rows in df_valid) the 1:1 match (pred-valid pair, "head(1)") with highest similarities across all three classes 
# NOTE need to sort based on all three smlrty cols to do correct Tp calc 
#      (if sort_values by on similartiy column would result in too many TPs- as then 1:1 pairs would contain also random matches where randomly CI_red is identical with CI_valid)
df_smltry_selmax = df_eval_records.groupby("id_valid").apply(lambda s: s.sort_values(["ci_smlrty","dam_smlrty","loc_smlrty_norm_pr"], ascending=False).head(1))
# # OLD  (makes too many 1:1 pairs as described in NOTE)
# mask = df_eval_records.groupby("id_valid").apply(lambda x: x==x["ci_smlrty"].max()).droplevel(0)
# df_smltry_selmax2 = df_eval_records.where(mask.ci_smlrty==mask.ci_smlrty.max()).dropna(how="all") # keep cases only which have highest similarity scores
# df_smltry_selmax2.reset_index(drop=True, inplace=True)


print("for each unique valid record keep only pred-valid pairs of highest similarity")


# iterate over the three entity classes (ci, damage, location) to assess LLM performance
for _, column_pred in zip(list_entity_valid, list_entity_pred):

    if column_pred == "ci_group_pred":

        # remove cases where no CI could be found (for Ci unlikelky, but more common for location or damage)
        df_valid_ci = df_valid[df_valid["ci1_group"].notnull()]
        df_pred_ci = df_pred[df_pred["infrastructure_group"].notnull()]

        # when no similarity could be calculated
        # ## FIXME move outside of loop
        # entries_with_no_similarity = df_eval_records.loc[df_eval_records["impact_sim_identical"].isna()]
        # print(f" --- Pred-valid pairs where no identical similarity score could be calculated: {len(entries_with_no_similarity)} ----")
        # print(entries_with_no_similarity[["impact_valid", "impact_pred", "impact_sim_identical", "impact_sim_cos", "impact_sim_pr", "citation"]])

        
        # TPs 
        tps = df_smltry_selmax.loc[df_smltry_selmax["ci_smlrty"] == 1]
        print(len(tps), len(df_smltry_selmax ), len(df_smltry_selmax[~df_smltry_selmax[["ci_pred", "ci_valid"]].isna().any(axis=1)]))
        
        
        # # FPs
        # --> make mask where records in df-eval record are identical to df_pred.columns (must be 1:1), 
        #     aplly mask on df_pred and substract from output all cases which are in TPs 
        # assert len(output) == fps_len
        
        # FNs
        ## missed docs
        df_valid_ci_pred_missed_docs = df_valid_ci[df_valid_ci["publication_id"].isin(df_pred["citation_id"]) == False]
        ## missed entries
        # extracts all duplicates (except first occurrence eg. id_Pred==537 occurs in df_smltry_selmax_p three times (1st case: TP or FP, 2nd and 3rd are FNs)
        df_valid_cases_missed_by_model = df_smltry_selmax[df_smltry_selmax.duplicated(subset="id_pred", keep="first")]
        ## OLD APPROACH: CI cases in valid set (for docs existing in both sets) - number of corectly predicted CI cases (Tps)
        ## no_valid_cases_missed_by_model  = len(df_valid_pred_same_docs["ci1_group"])  - len(df_smltry_selmax[df_smltry_selmax["impact_sim_identical"]==1])

        # ## TODO FIXME not sure if approach for df_valid_cases_missed_by_model based on df_smltry_selmax is correct
        # ##            as df_smltry_selmax contains only the cases of highest similarity for each case in df_valid (ie unique id_valid)
        # ##            can i then calc the number of missed cases by 
        
        # fns = pd.concat([df_valid_pred_missed_docs, df_valid_cases_missed_by_model], ignore_index=True, axis=0)

        # FIXME WORKAROUND for FP and FN calculation, but not extract respective cases (only numbers of FPs and FNs)
        fps_len = len(df_pred_ci["infrastructure_group"]) - tps.shape[0]
        fns_len = len(df_valid_ci["ci1_group"]) - tps.shape[0]

        print("tps", len(tps), " fps:", fps_len, " fns:", fns_len)
                    
        # performance scores
        recall_score = u.calc_recall(tps_no=len(tps), fns_no=fns_len) 
        precision_score = u.calc_precision(tps_no=len(tps), fps_no=fps_len)
        try:
            f1_score = u.calc_f1(precision=precision_score, recall=recall_score)
        except ZeroDivisionError:
            f1_score = 0.0

        print(f" ---------- Evaluation statistics: {column_pred}-----------")
        print(f"Recall: {recall_score}, Precision: {precision_score}, F1-score: {f1_score}")

        # saving
        dh.DataHandler().save_evaluation_results(column_pred, df_smltry_selmax, recall_score, precision_score, f1_score)


    if column_pred == "damage_pred":

        # remove cases where no CI could be found (for CI unlikelky, but more common for location or damage)
        df_valid_dam = df_valid[df_valid["ci1_damage"].notnull()]
        df_pred_dam = df_pred[df_pred["damage"].notnull()]  # when model gave NaN (then actually also corresponding df_valid record would be there NaN)

        # TPs 
        tps = df_smltry_selmax.loc[df_smltry_selmax["dam_smlrty"] >= cos_smlrty_thresh]

        # ## check if TPs calc correct
        # # tps_validmergedpred = df_valid.merge(df_pred, left_on=["ci1_damage"], right_on=["damage"], how="inner") ## ERROR as gives > 4000 entries
        # # assert len(tps) == len(tps_validmergedpred)

        # # FPs - model predicts condition wrongly (ie. predict condition when it is actually absent)
        # # get all valid. documents which were also used for LLM inference
        # df_valid_pred_same_docs = df_valid[df_valid["publication_id"].isin(df_pred["citation_id"])]
        # print(f"Doing evaluation based on {df_valid_pred_same_docs.publication_id.unique().__len__()} documents existing in both (valid.+pred. set)")
        # # get records where model predicted presence of impacts but they actually does not exist
        # fps = df_smltry_selmax.loc[df_smltry_selmax["impact_sim_identical"] < cos_smlrty_thresh]
        # # here as definition, that when simi=0 (or below threshold) then model predicted presences as false alarm
        # # TODO
        # # add also as Fps were model_pred case exist but no fitting_valid case could be found (during df_valid_pred pair generation in loop at begin of NB)
        # # df_pred selction needed

        # # FNs - CI impact cases not detected by model 
        # # NOTE: maybe FNs number is biased as wrong matches more likely as chunk-text (pred set) is longer than sentence text (valid set)
        # # WRONG? get all entries from df_valid_pred_same_docs where corresponding pred_record (in FPs) is missing

        # # get all documents in valid_set which does not occur in pred_set or where similarity is too low
        # ## missed docs
        # df_valid_pred_missed_docs = df_valid[df_valid["publication_id"].isin(df_pred["citation_id"]) == False]
        # print("Number of documents where model did not extract anything", df_valid_pred_missed_docs.shape)
        # ## missed entries
        # # extracts all duplicates (except first occurrence eg. id_Pred==537 occurs in df_smltry_selmax_p three times (1st case: TP or FP, 2nd and 3rd are FNs)
        # df_valid_cases_missed_by_model = df_smltry_selmax[df_smltry_selmax.duplicated(subset="id_pred", keep="first")]
        # ## OLD APPROACH: CI cases in valid set (for docs existing in both sets) - number of correctly predicted CI cases (Tps)
        # # no_valid_cases_missed_by_model  = len(df_valid_pred_same_docs["ci1_damage"])  - len(df_smltry_selmax["impact_valid"])
        # # no_valid_cases_missed_by_model  = len(df_valid_pred_same_docs["ci1_location"])  - len(df_smltry_selmax["impact_valid"])
        # fns = pd.concat([df_valid_pred_missed_docs, df_valid_cases_missed_by_model], ignore_index=True, axis=0)


        # FIXME WORKAROUND for FP and FN calculation, but not extract respective cases (only numbers of FPs and FNs)
        fps_len = len(df_pred_dam["damage"]) - tps.shape[0]  # that
        fns_len = len(df_valid_dam["ci1_damage"]) - tps.shape[0]

        print("tps", len(tps), " fps:", fps_len, " fns:", fns_len)

        # performance scores
        recall_score = u.calc_recall(tps_no=len(tps), fns_no=fns_len) 
        precision_score = u.calc_precision(tps_no=len(tps), fps_no=fps_len)
        try:
            f1_score = u.calc_f1(precision=precision_score, recall=recall_score)
        except ZeroDivisionError:
            f1_score = 0.0

        print(f" ---------- Evaluation statistics: {column_pred}-----------")
        print(f"Recall: {recall_score}, Precision: {precision_score}, F1-score: {f1_score}")
        
        ## saving
        dh.DataHandler().save_evaluation_results(column_pred, df_smltry_selmax, recall_score, precision_score, f1_score)
        

    if column_pred == "location_pred":

        # remove cases where no CI could be found (for CI unlikelky, but more common for location or damage)
        df_valid_loc = df_valid[df_valid["ci1_location"].notnull()]
        df_pred_loc = df_pred[df_pred["location"].notnull()]  # when model gave NaN (then actually also corresponding df_valid record would be there NaN)

        # TPs 
        tps = df_smltry_selmax.loc[df_smltry_selmax["loc_smlrty_norm_pr"] >= norm_pr_smlrty_thresh]

        # ## check if TPs calc correct
        # # tps_validmergedpred = df_valid.merge(df_pred, left_on=["ci1_location"], right_on=["location"], how="inner") ## ERROR as gives > 4000 entries
        # # assert len(tps) == len(tps_validmergedpred)

        # # FPs - model predicts condition wrongly (ie. predict condition when it is actually absent)
        # # get all valid. documents which were also used for LLM inference
        # df_valid_pred_same_docs = df_valid[df_valid["publication_id"].isin(df_pred["citation_id"])]
        # print(f"Doing evaluation based on {df_valid_pred_same_docs.publication_id.unique().__len__()} documents existing in both (valid.+pred. set)")
        # # get records where model predicted presence of impacts but they actually does not exist
        # fps = df_smltry_selmax.loc[df_smltry_selmax["impact_sim_identical"] < cos_smlrty_thresh]
        # # here as definition, that when simi=0 (or below threshold) then model predicted presences as false alarm
        # # TODO
        # # add also as Fps were model_pred case exist but no fitting_valid case could be found (during df_valid_pred pair generation in loop at begin of NB)
        # # df_pred selction needed

        # # FNs - CI impact cases not detected by model 
        # # NOTE: maybe FNs number is biased as wrong matches more likely as chunk-text (pred set) is longer than sentence text (valid set)
        # # WRONG? get all entries from df_valid_pred_same_docs where corresponding pred_record (in FPs) is missing

        # # get all documents in valid_set which does not occur in pred_set or where similarity is too low
        # ## missed docs
        # df_valid_pred_missed_docs = df_valid[df_valid["publication_id"].isin(df_pred["citation_id"]) == False]
        # print("Number of documents where model did not extract anything", df_valid_pred_missed_docs.shape)
        # ## missed entries
        # # extracts all duplicates (except first occurrence eg. id_Pred==537 occurs in df_smltry_selmax_p three times (1st case: TP or FP, 2nd and 3rd are FNs)
        # df_valid_cases_missed_by_model = df_smltry_selmax[df_smltry_selmax.duplicated(subset="id_pred", keep="first")]
        # ## OLD APPROACH: CI cases in valid set (for docs existing in both sets) - number of correctly predicted CI cases (Tps)
        # # no_valid_cases_missed_by_model  = len(df_valid_pred_same_docs["ci1_damage"])  - len(df_smltry_selmax["impact_valid"])
        # # no_valid_cases_missed_by_model  = len(df_valid_pred_same_docs["ci1_location"])  - len(df_smltry_selmax["impact_valid"])
        # fns = pd.concat([df_valid_pred_missed_docs, df_valid_cases_missed_by_model], ignore_index=True, axis=0)
        
        # FIXME WORKAROUND for FP and FN calculation, but not extract respective cases (only numbers of FPs and FNs)
        fps_len = len(df_pred_loc["location"]) - tps.shape[0]
        fns_len = len(df_valid_loc["ci1_location"]) - tps.shape[0]
        
        print("tps", len(tps), " fps:", fps_len, " fns:", fns_len)

        # performance scores
        recall_score = u.calc_recall(tps_no=len(tps),fns_no=fns_len) 
        precision_score = u.calc_precision(tps_no=len(tps), fps_no=fps_len)
        try:
            f1_score = u.calc_f1(precision=precision_score, recall=recall_score)
        except ZeroDivisionError:
            f1_score = 0.0
        
        print(f" ---------- Evaluation statistics: {column_pred}-----------")
        print(f"Recall: {recall_score}, Precision: {precision_score}, F1-score: {f1_score}")

        ## saving
        dh.DataHandler().save_evaluation_results(column_pred, df_smltry_selmax, recall_score, precision_score, f1_score)


### LAMA 3 + NER + GeoLLM (step 2)

# 43 43 22  - 24 docs
# tps 43  fps: 104  fns: 38
#  ---------- Evaluation statistics: ci_group_pred-----------
# Recall: 0.5308641975308642, Precision: 0.2925170068027211, F1-score: 0.37719298245614036
# Saving evaluation statistics, distribution plots, and scores to  ci_group_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 8  fps: 91  fns: 67
#  ---------- Evaluation statistics: damage_pred-----------
# Recall: 0.10666666666666667, Precision: 0.08080808080808081, F1-score: 0.09195402298850575
# Saving evaluation statistics, distribution plots, and scores to  damage_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 32  fps: 115  fns: 49
#  ---------- Evaluation statistics: location_pred-----------
# Recall: 0.3950617283950617, Precision: 0.21768707482993196, F1-score: 0.2807017543859649
# Saving evaluation statistics, distribution plots, and scores to  location_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]


### LAMA 3 + NER + GeoLLM (step 1) - no countries (only regions, cities, etc)
## not better than with countries (slight increase in recall+precision for LOC )

### LAMA 3 + NER + GeoLLM (step 1)  - 24 docs
# or each unique valid record keep only pred-valid pairs of highest similarity
# 39 46 46
# tps 39  fps: 186  fns: 49
#  ---------- Evaluation statistics: ci_group_pred-----------
# Recall: 0.4431818181818182, Precision: 0.17333333333333334, F1-score: 0.24920127795527158
# Saving evaluation statistics, distribution plots, and scores to  ci_group_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 16  fps: 176  fns: 66
#  ---------- Evaluation statistics: damage_pred-----------
# Recall: 0.1951219512195122, Precision: 0.08333333333333333, F1-score: 0.11678832116788321
# Saving evaluation statistics, distribution plots, and scores to  damage_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 26  fps: 199  fns: 62
# ---------- Evaluation statistics: location_pred-----------
# Recall: 0.29545454545454547, Precision: 0.11555555555555555, F1-score: 0.16613418530351434
# Saving evaluation statistics, distribution plots, and scores to  location_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json] 

### LAMA 3 + NER
# 15 33
# tps 15  fps: 27  fns: 18
#  ---------- Evaluation statistics: ci_group_pred-----------
# Recall: 0.45454545454545453, Precision: 0.35714285714285715, F1-score: 0.4
# Saving evaluation statistics, distribution plots, and scores to  ci_group_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 11  fps: 31  fns: 19
#  ---------- Evaluation statistics: damage_pred-----------
# Recall: 0.36666666666666664, Precision: 0.2619047619047619, F1-score: 0.3055555555555555
# Saving evaluation statistics, distribution plots, and scores to  damage_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 10  fps: 32  fns: 23
#  ---------- Evaluation statistics: location_pred-----------
# Recall: 0.30303030303030304, Precision: 0.23809523809523808, F1-score: 0.26666666666666666

## Lama 3 # 1 doc Koks 2022
# 20 33
# tps 20  fps: 30  fns: 13
#  ---------- Evaluation statistics: ci_group_pred-----------
# Recall: 0.6060606060606061, Precision: 0.4, F1-score: 0.4819277108433735
# Saving evaluation statistics, distribution plots, and scores to  ci_group_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# ps 12  fps: 38  fns: 18
#  ---------- Evaluation statistics: damage_pred-----------
# Recall: 0.4, Precision: 0.24, F1-score: 0.3
# Saving evaluation statistics, distribution plots, and scores to  damage_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# ps 12  fps: 38  fns: 21
#  ---------- Evaluation statistics: location_pred-----------
# Recall: 0.36363636363636365, Precision: 0.24, F1-score: 0.2891566265060241
# Saving evaluation statistics, distribution plots, and scores to  location_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]



# llm_1_updprompt_dNER.csv
# 45 67
# tps 45  fps: 1115  fns: 22
#  ---------- Evaluation statistics: ci_group_pred-----------
# Recall: 0.6716417910447762, Precision: 0.03879310344827586, F1-score: 0.07334963325183375
# tps 14  fps: 1146  fns: 47
#  ---------- Evaluation statistics: damage_pred-----------
# Recall: 0.22950819672131148, Precision: 0.01206896551724138, F1-score: 0.022932022932022934
# tps 11  fps: 1149  fns: 43
#  ---------- Evaluation statistics: location_pred-----------
# Recall: 0.2037037037037037, Precision: 0.009482758620689655, F1-score: 0.018121911037891267


 --- For each unique valid case (unique combi: [ci_valid, damage_valid, location_valid, sentence_text]) calculate similarity ---
Using 100% match for CI types based on subgroups
Using cosine similarity threshold for damages 0.7
Using normalized partial ratio similarity threshold for locations 0.7
Record: 0 / 203


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Record: 1 / 203


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Record: 202 / 203


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

for each unique valid record keep only pred-valid pairs of highest similarity
44 53 53
tps 44  fps: 107  fns: 50
 ---------- Evaluation statistics: ci_group_pred-----------
Recall: 0.46808510638297873, Precision: 0.2913907284768212, F1-score: 0.3591836734693878
Saving evaluation statistics, distribution plots, and scores to  ci_group_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
tps 19  fps: 121  fns: 68
 ---------- Evaluation statistics: damage_pred-----------
Recall: 0.21839080459770116, Precision: 0.1357142857142857, F1-score: 0.16740088105726872
Saving evaluation statistics, distribution plots, and scores to  damage_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
tps 25  fps: 126  fns: 69
 ---------- Evaluation statistics: location_pred-----------
Recall: 0.26595744680851063, Precision: 0.16556291390728478, F1-score: 0.20408163265306123
Saving evaluation statistics, distribution plots, and scores to  location_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]


In [ ]:
## whitout FAC-based records
# for each unique valid record keep only pred-valid pairs of highest similarity
# 44 53 53
# tps 44  fps: 107  fns: 50
#  ---------- Evaluation statistics: ci_group_pred-----------
# Recall: 0.46808510638297873, Precision: 0.2913907284768212, F1-score: 0.3591836734693878
# Saving evaluation statistics, distribution plots, and scores to  ci_group_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 19  fps: 121  fns: 68
#  ---------- Evaluation statistics: damage_pred-----------
# Recall: 0.21839080459770116, Precision: 0.1357142857142857, F1-score: 0.16740088105726872
# Saving evaluation statistics, distribution plots, and scores to  damage_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 25  fps: 126  fns: 69
#  ---------- Evaluation statistics: location_pred-----------
# Recall: 0.26595744680851063, Precision: 0.16556291390728478, F1-score: 0.20408163265306123
# Saving evaluation statistics, distribution plots, and scores to  location_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]

for each unique valid record keep only pred-valid pairs of highest similarity
44 53 53
tps 44  fps: 107  fns: 50
 ---------- Evaluation statistics: ci_group_pred-----------
Recall: 0.46808510638297873, Precision: 0.2913907284768212, F1-score: 0.3591836734693878
Saving evaluation statistics, distribution plots, and scores to  ci_group_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
tps 19  fps: 121  fns: 68
 ---------- Evaluation statistics: damage_pred-----------
Recall: 0.21839080459770116, Precision: 0.1357142857142857, F1-score: 0.16740088105726872
Saving evaluation statistics, distribution plots, and scores to  damage_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
tps 25  fps: 126  fns: 69
 ---------- Evaluation statistics: location_pred-----------
Recall: 0.26595744680851063, Precision: 0.16556291390728478, F1-score: 0.20408163265306123
Saving evaluation statistics, distribution plots, and scores to  location_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]


#### ISSUE: some pred-valid matches are wrongly matched in Text-Simil.based eval
--> (probably bc no better pred_entry for corresponding valid_entry exists) e,g, 
* pred: "in ahr valley some roads were destroyed. the railways in Germany were damaged" - roads,destroyed, ahr valley
* valid: "the railways in Germany were damaged" - railways, damaged, Germany

IDEA. do document-wise comparison or based on spatial agg. --> later maybe better da 

In [43]:
df_pred.to_csv(f"df_pred_{step}_why_ci_dam_loc_bad.csv")
df_smltry_selmax.to_csv(f"df_smlrty_max_{step}_why_ci_dam_loc_bad.csv")

## EVAL why performance is not good LLAMA 3


In [44]:
df_smltry_selmax[["citation", "sentence_text_valid", "id_pred", "id_valid", "ci_pred", "ci_valid", "ci_smlrty", "dam_pred", "dam_valid", "dam_smlrty", "loc_pred", "loc_valid", "loc_smlrty", "loc_smlrty_norm_pr"]].head(5)

KeyError: "['id_valid'] not in index"

#### FIXME: find out which cases model predicted existence, but not in valid DS - maybe due that valid DS is incomppete?

In [ ]:
df_pred.info()

In [ ]:
## fix FPs 

## get all pred cases which 
# rows in df_valid where sentence_reference appears as substring in at least one df_pred.chunk_text
chunk_texts = df_pred["chunk_text"].dropna().astype(str)

df_valid_2 = df_valid[
    df_valid["sentence_reference"].fillna("").astype(str).apply(
        lambda s: any(s and s in chunk for chunk in chunk_texts)
    )
]

df_valid_2 # .shape (28, 7)

In [ ]:
# cases where pred-case exist but no fitting valid case could be found based on sentence_reference
df_pred_not_in_valid = df_pred[df_pred['chunk_text'].str.contains('|'.join(df_valid["sentence_reference"]), regex=True)]
print(df_pred_not_in_valid.id_pred.value_counts())
df_pred_not_in_valid.head(10)

# TODO TODO
## documents where model found many CI cases as false-alarms:
# maybe i need to recheck those docs and make df_valid more complete
# print(df_pred_not_in_valid.groupby("citation_id").count())
# citation_id                                                            
# ABC 2024                        4
# Containerlift 2024             24
# European Investment Bank 2025  21
# Ferlita 2023                   22
# Koks 2022                      10


In [ ]:
df_valid

#### FIXME: FPs and FNs

In [ ]:
## TPs + FNs should be == len(df_valid.ci) == 67
        
# TPs 
tps = df_smltry_selmax.loc[df_smltry_selmax["impact_sim_identical"] == 1]

# FNs
df_valid_pred_missed_docs = df_valid[df_valid["publication_id"].isin(df_pred["citation_id"]) == False]
df_valid_cases_missed_by_model = df_smltry_selmax[df_smltry_selmax.duplicated(subset="id_pred", keep="first")]
fns = pd.concat([df_valid_pred_missed_docs, df_valid_cases_missed_by_model], ignore_index=True, axis=0)

print(tps.shape[0], fns.shape[0])
print(tps.shape[0] + fns.shape[0])

# --> 8 cases in FNs are too definitly too much --> fix FN calculation



## FPs should be == len(df_pred.ci) - TPs

## FPs
fps = df_smltry_selmax.loc[df_smltry_selmax["impact_sim_identical"] == 0]

print(len(df_pred.infrastructure_type), tps.shape[0], fps.shape[0])
print(len(df_pred.infrastructure_type) - tps.shape[0])




In [ ]:
# TODO fix FPs
# get records where model predicted presence of impacts but they actually does not exist
# here as definition, that when simi=0 (or below threshold) then model predicted wrongly
df_smltry_not_sim = df_smltry_selmax.loc[df_smltry_selmax["impact_sim_identical"] == 0]

# TODO
# add also as Fps were model_pred case exist but no fitting_vlaid case could be found

# idea: 
# get all df_pred cases where chunk text not occurs in valid.sentece_text

df_pred_not_in_valid = df_pred[df_pred["chunk_text"].isin(df_valid["sentence_reference"])== False]
print(df_pred.shape, df_pred_not_in_valid.shape)
# df_pred_not_in_valid

In [ ]:
# df_valid__pred_no_thresh.id_pred.nunique()
df_pred_valid_no_thresh.id_pred.nunique()

In [ ]:
# df_valid__pred_no_thresh.info()
# df_pred_valid_no_thresh.info()  # 67 valid * 921 pred = 61707
# df_pred_valid_no_thresh.drop("chunk_text_pred", axis=1).sort_values("id_pred").iloc[0:100]
# df_pred_valid_no_thresh.groupby("id_pred").first().sort_values("text_similarity", ascending=False).iloc[0:100]
# df_valid__pred_no_thresh.groupby("id_valid").first().sort_values("text_similarity", ascending=False).iloc[0:100]
#df_valid__pred_no_thresh.sort_values("id_valid", ascending=False).sort_values("text_similarity", ascending=False).iloc[0:100]
#df_valid__pred_no_thresh.groupby("id_valid").first().sort_values("text_similarity", ascending=False).iloc[0:100]
df_valid__pred_no_thresh.groupby("id_valid").apply(lambda x: x.loc[x["text_similarity"].idxmax()])


In [ ]:
# fns

In [ ]:
## FNs 
df_smltry_selmax.loc[df_smltry_selmax.duplicated("id_pred")]
## --> ISSUE: this df (cases of highest sim) should NOT have duplicated cases of predictions -> maybe have to group based on id_pred and not id_valid

## try to fix issue
## --> currently i think this should group based on valid cases to measure were model predicted the same or missed info (ie FNs)
# df_smltry_selmax_p = df_smltry_selmax
# mask of rows with highest similarity score for each set of preds with unique valid case (droplevel(0) remove multiindex)
mask = df_smltry_all.loc[df_smltry_all.id_valid==37].groupby("id_valid").apply(lambda x: x==x["impact_sim_identical"].max()).droplevel(0)
df_smltry_selmax_p = df_smltry_all.where(mask.impact_sim_identical==mask.impact_sim_identical.max()).dropna(how="all") # drop cases which have not highest similairty score
# df_smltry_selmax_p = df_smltry_all.groupby("id_valid").apply(lambda x: x.loc[x["impact_sim_identical"].idxmax()]) # 52 cases
# df_smltry_selmax_p = df_smltry_all.groupby("id_pred").apply(lambda x: x.loc[x["impact_sim_identical"].idxmax()]) # 175 cases
df_smltry_selmax_p.reset_index(drop=True, inplace=True)

## FIXME  df_smltry_all.groupby("id_valid"): should it has duplicated cases of id_pred ? - i dont think so! 
#  bc it means that there model missed cases in valid_set
## --> so all duplicated cases (except one-this is TP or FP) are actual FNs
print(df_smltry_selmax_p.info())
print(df_smltry_selmax_p.id_pred.nunique()  )  # should be len of df
print(df_smltry_selmax_p.duplicated().sum())


# FNs: extracts all duplicates (except first occurrence eg. id_Pred==537 occurs in df_smltry_selmax_p three times (1st case: TP or FP, 2nd and 3rd are FNs)
fns = df_smltry_selmax_p[df_smltry_selmax_p.duplicated(subset="id_pred", keep="first")]

print(fns.info())
fns.id_pred.value_counts()


In [ ]:
# return all cases which has max sim also when max score is shared by multiple rows 
# df_smltry_all.loc[df_smltry_all.groupby("id_valid").transform(lambda x: x==x.max()).astype('bool')].shape
mask = df_smltry_all.loc[df_smltry_all.id_valid==37].groupby("id_valid").apply(lambda x: x==x["impact_sim_identical"].max())
mask = mask.droplevel(0)
#.transform(lambda x: x==x.max())
tt = df_smltry_all.loc[df_smltry_all.id_valid==37]#
tt = tt.where(mask.impact_sim_identical==mask.impact_sim_identical.max()).dropna(how="all") # drop cases which have not highest similairty score

# tt[mask]

# would return only first case of max sim:
#df_smltry_all.loc[df_smltry_all.id_valid==37].groupby("id_valid").apply(lambda x: x.loc[x["impact_sim_identical"].idxmax()]) #


In [ ]:
# FPs. 
print("False alarms (where model predicted ci but no corresponding valid case exists)", 
      len(df_pred["infrastructure_type"])  - len(df_smltry_selmax[df_smltry_selmax["impact_sim_identical"]== 1, "ci_group_pred"])
    )
# Get FPs - cases where model predicted presence of CI (but actually it is absent in valid set)
tt = df_pred.merge(
    # FIXME issue that df_pred_valid_all contains some duplicates where id_pred identical but not valid_entries
    df_smltry_selmax.drop_duplicates(), # safety: make sure that merging is done on 1:1 match
    left_on="id_pred",#["citation_id", "chunk_id","infrastructure_type", "damage", "location"], 
    right_on="id_pred",#["citation_id", "chunk_id_pred", "ci_pred", "damage_pred", "location_pred"],
    how="left",
    indicator=True    # return an extra column indicating which table the row was from.
)
tt = tt.loc[tt["_merge"] == "left_only"].drop(columns=["_merge"])
print("False positives (model predicted CI but no corresponding valid case exists):", len(tt))

In [ ]:
print(df_pred.shape[0])
# print(df_pred_valid_all.info())
print(tt.info())

In [ ]:
df_pred#["infrastructure_type"]

In [ ]:
df_smltry_selmax.info()

In [ ]:
# df_smltry_selmax["impact_sim_identical"] < cos_smlrty_thresh

In [ ]:
# len(df_valid_pred_same_docs["ci1_group"]) 

In [ ]:
# TODO fix FNs
print(df_valid_pred_same_docs.info())
print(df_smltry_selmax[df_smltry_selmax["impact_sim_identical"]==1].info())
# --> FNS should be  23
len(df_valid_pred_same_docs["ci1_group"])  - len(df_smltry_selmax[df_smltry_selmax["impact_sim_identical"]==1]["impact_valid"])

In [ ]:
# #df_valid_pred_same_docs["id_valid"] = df_valid_pred_same_docs.apply(lambda x: f"{x['ci1_group']}_{x['ci1_damage']}_{x['ci1_location']}_{x['sentence_text_valid'][:50]}", axis=1)
# print(df_valid_pred_same_docs["id_valid"].unique().__len__())
# print(df_valid_pred_same_docs.shape[0])
# ## --> check why electricity_others_outages_nan = 3  - (seems correct as sentences_ref are diff). airports_affected_Malaga area=2 are not unique
# df_valid_pred_same_docs[df_valid_pred_same_docs["id_valid"] == "airports_affected_Malaga area"]

# Improve similarity
As all similarity measures - no matter which emebdding model or kind of cosine similarity measure) were not sufficient eg. port ~ power to similar to port~harbor

Thus, it might be better to first group ci impacts into subgroups e.g .based on HARCI-EU categories,as some kind of postprocessing step before applying the similarity measurements



In [ ]:
# s = "dyke" to s2 = "levee", s3 = "dam"
# bge-m3: 0.48  0.54
# all-MIniLM-L6-v2: 0.34 , 0.36  (similar all-mpnet-base-v2)
# gensim word2vec: 0.39 0.40


# s1 = "aviation" s2 = "air traffic"
# word vector spacy: 0.45
# contextual vector spacy: 0.68
# bge-m3: 0.76
# all-MIniLM-L6-v2: xx  (all-mpnet-base-v2: 0.79)
# gensim word2vec: 


# s1 = "power" s2 = "electricity"
# word vector spacy: 0.61
# contextual vector spacy: 0.66
# bge-m3: 
# all-MIniLM-L6-v2: xx   (all-mpnet-base-v2: 0.43)
# gensim word2vec: 0.58


# s1 = "electricity infrastructure" s2 = "electricity"
# word vector spacy: 0.87
# contextual vector spacy: 0.71
# bge-m3: 
# all-MIniLM-L6-v2:   xx  (all-mpnet-base-v2: 0.63)
# gensim word2vec: 


# s1 = "transportation" s2 = "transport infrastructure"
# word vector spacy: 
# contextual vector spacy: 
# bge-m3: 
# all-MIniLM-L6-v2:   xx  (all-mpnet-base-v2: 0.84)
# gensim word2vec: 

# s1 = "port" s2 = "power"  s3= harbour
# bge-m3: 0.58, 0.50
# all-MIniLM-L6-v2: 0.33 , 0.56  (similar all-mpnet-base-v2)
# gensim word2vec: 0.14 0.59

# s1 = "electricity" s2 = "transportation" 
# bge-m3:  0.64
# all-MIniLM-L6-v2:   (all-mpnet-base-v2: 0.47)
# gensim word2vec: 0.33

In [ ]:
# # print(cos_sim(model_scs["transportation"], model_scs["transport infrastructure"]))
# # print(cos_sim(model_scs["electricity infrastructure"], model_scs["electricity"]))
# # print(cos_sim(model_scs["power plant"], model_scs["electricity"]))
# print(cos_sim(model_scs["power"], model_scs["electricity"]))
# print(cos_sim(model_scs["aviation"], model_scs["air traffic"]))
# # identical to model_scs.similarity("port", "power"))


# # similarity_score = 1-distance.cosine(model.encode([s1])[0], model.encode([s2])[0])

### Analyse evaluation results 


In [ ]:
df_smltry_selmax#.info()

In [ ]:
## find out for which docs model performed bad (or good)
## based on this info try to improve model 

df_smltry_selmax.dropna(subset=["impact_sim_cos"]).groupby("citation").apply(lambda x: x.loc[x["impact_sim_cos"].idxmax()]).sort_values(by="impact_sim_cos", ascending=True)
## check EFE, Wilson, European Investment Bank, Containerlift, Lloyds List, Gilbody Dickerson


In [ ]:
## check entries of worst performace docs for damage
df_smltry_selmax.loc[df_smltry_selmax["citation"].isin(["Khazai 2023", "ABC 2024", "Containerlift 2024", "Lloyds List 2024", "Ferlita 2023"])]

In [ ]:
## check entries of worst performance docs for Ci tyes
df_smltry_selmax.loc[df_smltry_selmax["citation"].isin(["EFE 2024", "Containerlift 2024", "Lloyds List 2024", "Wilson 2024", "Gilbody Dickerson 2024", "European Investment Bank 2025"])].head(50)


## For each validation entry, search for all prediction cases of the same chunk 

In [ ]:
## get same impact entries
list_entity_valid = ["ci1_type", "ci1_damage", "ci1_location"]
list_entity_pred = ["infrastructure_type", "damage", "location"]


for entity_valid, entity_pred in zip(list_entity_valid, list_entity_pred):

    print(f" --------- Processing column pair: {entity_valid} - {entity_pred} ------------")
    
    df_valid_pred_all = pd.DataFrame()
    citations_list = []

    ## for each validation record
    for i in range(len(df_valid)):
        
        highest_similarity_score = 0.00
        
        ## needed to traceback info when entry is missing in pred. DS
        # chunk_id_value_valid = df_valid.chunk_id[i]

        # select nth validation record and check that it has value
        df_valid_entry = df_valid.iloc[i]
        if df_valid_entry[entity_valid] is np.nan:
            continue
        
        citation_str = df_valid_entry.publication_id
        citations_list.append(citation_str)


        # get all corresponding prediction records
        df_pred_entries = df_pred[df_pred["citation_id"].isin([citation_str])]

        #  handle on NANs
        df_pred_entries[entity_pred] = np.where(df_pred_entries[entity_pred].isna(), "nan", df_pred_entries[entity_pred])
        # df_pred_entries[entity_pred] = df_pred_entries[entity_pred].astype(str)
        # remove double whitespaces
        # df_pred_doc[entity_pred] = df_pred_doc[entity_pred].replace("  ", " ")
        # df_valid_entries[entity_valid] = df_valid_entries[entity_valid].replace("  ", " ")


        # vector of validiation entry 
        valid_impact = df_valid_entry[entity_valid]
        valid_vec = nlp(valid_impact).vector

        # print(" ------- Searching for citation:", citation_str, " in predictions ------- ")

        # Compute similarity between each validation CI impact case and all potential predicted CI impact cases (cross-product)
        for j in range(len(df_pred_entries[entity_pred])):

            if df_pred_entries[entity_pred].iloc[j] == "nan":
                continue

            pred_impact = df_pred_entries[entity_pred].iloc[j]

            pred_vec = nlp(pred_impact).vector
            similarity_score = u.cosine_similarity(valid_vec, pred_vec)  # 0-1 value, the higher the more similar
            # print(f"Similarity {i}-{j}: {similarity_score}")

            # print(f"Searching for highest similarity ... ")
            ## get only pair with highest similarity
            if similarity_score > highest_similarity_score:
                
                highest_similarity_score = similarity_score
                
                dict_pair = {
                    "impact_valid": valid_impact, 
                    "impact_pred": pred_impact, 
                    "similarity": highest_similarity_score,
                    "citation": citation_str,
                    "chunk_id_pred": (df_pred.chunk_id[i],  df_pred.chunk_id[j])
                }
            else:
                continue

        df_valid_pred_all = pd.concat([df_valid_pred_all, pd.DataFrame([dict_pair])], ignore_index=True)


    print(f" ---------- Evaluation summary statistics - {entity_pred}: -----------")
    print(df_valid_pred_all.similarity.describe())

    

    SIMILARITY_FILENAME = f'{entity_pred}_{SIMILARITY_LLM_FILENAME}'
    SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

    print("Saving evaluation statistics, distribution plots, and scores to ", SIMILARITY_FILEPATH.stem, "[.parquet, _stats.json]")
    with open(SIMILARITY_FILEPATH, 'w') as f:
        # results
        df_valid_pred_all.to_csv(SIMILARITY_FILEPATH.with_suffix('.csv'), index=False)
        df_valid_pred_all_pyarrow = pa.Table.from_pandas(df_valid_pred_all)
        pq.write_table(df_valid_pred_all_pyarrow, SIMILARITY_FILEPATH)   
        #   summary statistics
        df_valid_pred_all_stats = df_valid_pred_all.describe()
        f = SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_stats.json"  
        df_valid_pred_all_stats.to_json(f, indent=4)
        # distribution plots
        df_valid_pred_all.similarity.hist(bins=100).to_file(SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_hist.png")



    # cos_smlrty_thresh = 0.75
    # df_valid_pred_all['is_similar'] = df_valid_pred_all['similarity'] <= cos_smlrty_thresh
    # print(f"Number of similar impact cases (similarity >= {cos_smlrty_thresh}): {df_valid_pred_all['is_similar'].sum()} out of {len(df_valid_pred_all)}\n")

    # df_valid_pred_all =  df_valid_pred_all[df_valid_pred_all['similarity'] <= cos_smlrty_thresh]

    # SIMILARITY_FILENAME = f'{entity_pred}_lower75_{SIMILARITY_LLM_FILENAME}'
    # SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

    # with open(SIMILARITY_FILEPATH, 'w') as f:
    #     # results
    #     df_valid_pred_all.to_csv(SIMILARITY_FILEPATH.with_suffix('.csv'), index=False)
    #     df_valid_pred_all_pyarrow = pa.Table.from_pandas(df_valid_pred_all)
    #     pq.write_table(df_valid_pred_all_pyarrow, SIMILARITY_FILEPATH)  
    #     #   summary statistics
    #     df_valid_pred_all_stats = df_valid_pred_all.describe()
    #     f = SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_stats.json"  
    #     df_valid_pred_all_stats.to_json(f, indent=4)


In [ ]:
# df_valid_pred_all[df_valid_pred_all['similarity'] <= 0.75]

# df_valid_pred_all.similarity.hist(bins=100)

In [ ]:
list_entity_pred

In [ ]:
LLM_DATA_FILEPATH

### Load parquet file

In [ ]:

list_entity_pred = ["infrastructure_type", "damage", "location"]

In [ ]:
entity_pred = "infrastructure_type"
SIMILARITY_FILENAME = f'llm1_similarity_{entity_pred}_75.parquet'
SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    df = pd.read_parquet(SIMILARITY_FILEPATH, engine='pyarrow')
    display(df)

## Archive

In [ ]:
## Aim 
## for all identical valid entries ie. with same [ci_valid	damage_valid	location_valid	sentence_text_valid]
## get the match to pred_entity with highest similarity

In [ ]:
    # ## calc for each entry with the same chunk_text the similarity between valid_impact and pred_impact
    # ## means we calc also the False Negatives (ie. where valid entry exists but no prediction)


    # # iterate over groups of entities which refer to the same valid case (i.e. which are identical in valid_columns)
    # # TODO iterate over unqiue cases in df_valid (instead of using grouper)
    # grouper = df_pred_valid_all[["ci_valid", "damage_valid", "location_valid", "sentence_text_valid"]].drop_duplicates()
    # for group in grouper.itertuples():
    #     df_pred_valid_group = df_pred_valid_all[df_pred_valid_all[["ci_valid", "damage_valid", "location_valid", "sentence_text_valid"]] == group[["ci_valid", "damage_valid", "location_valid", "sentence_text_valid"]]]

    #     # calc. similarities to pred_entities
    #     for i, entry in df_pred_valid_group.iterrows():

    #         highest_similarity_score = 0 

    #         if entry[entity_pred].iloc[i] == "nan":
    #             continue
            
    #         # calc embeddings
    #         pred_impact = entry[entity_pred].iloc[i]
    #         pred_vec = nlp(pred_impact).vector

    #         valid_impact = entry[entity_valid].iloc[i] # is unique for each group
    #         valid_vec = nlp(valid_impact).vector
    #         print(valid_impact, "valid_impact")
            
    #         similarity_score = u.cosine_similarity(valid_vec, pred_vec)  # 0-1 value, the higher the more similar
    #         # print(f"Similarity {i}-{j}: {similarity_score}")

    #         ## return only pred-valid-pair with highest similarity
    #         if similarity_score > highest_similarity_score:
                
    #             highest_similarity_score = similarity_score
                
    #             entry["impact_similarity"] = highest_similarity_score

    # ## FNs
    # # # calc FN when valid_info exists but not corresponding pred_info
    # ## number of FNs is small due that wrong matching with any chunk-text is more likely due to its text size comapred sentence-level (valid set) 
    # elif entry[entity_pred] is np.nan:
    #     similarity_score = 0
    #     dict_pair = {
    #         "impact_valid": valid_impact, 
    #         "impact_pred": pred_impact, 
    #         "impact_similarity": similarity_score,
    #         "tp_tn_fp_fn": "fn",
    #         "citation": entry.citation_id,
    #         "chunk_text_pred": entry.chunk_text_pred,
    #         "sentence_text_valid": entry.sentence_text_valid,
    #         }
    #     df_smltry_selmax = pd.concat([df_smltry_selmax, pd.DataFrame([dict_pair])], ignore_index=True)

    # ## FPs
    # elif entry[entity_valid] is np.nan:
    #     similarity_score = 0
    #     dict_pair = {
    #         "impact_valid": valid_impact, 
    #         "impact_pred": pred_impact, 
    #         "impact_similarity": similarity_score,
    #         "tp_tn_fp_fn": "fp",
    #         "citation": entry.citation_id,
    #         "chunk_text_pred": entry.chunk_text_pred,
    #         "sentence_text_valid": entry.sentence_text_valid,
    #         }
    #     df_smltry_selmax = pd.concat([df_smltry_selmax, pd.DataFrame([dict_pair])], ignore_index=True)




In [ ]:
# ## get same impact entries
# list_entity_valid = ["ci1_type", "ci1_damage", "ci1_location"]
# list_entity_pred = ["infrastructure_type", "damage", "location"]



## iterate over predictions and search for each prediction reocrds for corresponding valid cases 

# for entity_valid, entity_pred in zip(list_entity_valid, list_entity_pred):

#     print(f" --------- Processing column pair: {entity_valid} - {entity_pred} ------------")
    
#     df_valid_pred_all = pd.DataFrame()
#     citations_list = []

#     ## for each validation record
#     for i in range(len(df_valid)):
        
#         highest_similarity_score = 0.00
        
#         ## needed to traceback info when entry is missing in pred. DS
#         # chunk_id_value_valid = df_valid.chunk_id[i]

#         # select nth validation record
#         df_valid_entry = df_valid.iloc[i]
#         citation_str = df_valid_entry.publication_id
#         citations_list.append(citation_str)
#         print(" ------- Searching for citation:", citation_str, " in predictions ------- ")


#         # get all corresponding prediction records
#         df_pred_entries = df_pred[df_pred["citation_id"].isin([citation_str])]
#         #  handle on NANs
#         df_pred_entries[entity_pred] = np.where(df_pred_entries[entity_pred].isna(), "nan", df_pred_entries[entity_pred])
#         # df_pred_entries[entity_pred] = df_pred_entries[entity_pred].astype(str)
#         # remove double whitespaces
#         # df_pred_doc[entity_pred] = df_pred_doc[entity_pred].replace("  ", " ")
#         # df_valid_entries[entity_valid] = df_valid_entries[entity_valid].replace("  ", " ")

#         # skip when validation entry ha no value
#         if df_valid_entry[entity_valid] is np.nan:
#             continue

#         # vector of validiation entry 
#         valid_impact = df_valid_entry[entity_valid]
#         valid_vec = nlp(valid_impact).vector


#         # Compute similarity between each predicted impact case and all potential validation impact cases (cross-product)
#         # print(f"Searching for highest similarity of`{pred_impact}` in validation set ... ")
#         for j in range(len(df_pred_entries[entity_pred])):

#             if df_pred_entries[entity_pred].iloc[j] == "nan":
#                 continue

#             pred_impact = df_pred_entries[entity_pred].iloc[j]

#             pred_vec = nlp(pred_impact).vector
#             similarity_score = u.cosine_similarity(valid_vec, pred_vec)  # 0-1 value, the higher the more similar
#             # print(f"Similarity {i}-{j}: {similarity_score}")

#             ## get only pair with highest similarity
#             if similarity_score > highest_similarity_score:
                
#                 highest_similarity_score = similarity_score
                
#                 dict_pair = {
#                     "impact_valid": valid_impact, 
#                     "impact_pred": pred_impact, 
#                     "similarity": highest_similarity_score,
#                     "citation": citation_str,
#                     "chunk_id_pred": df_pred.chunk_id[i]
#                 }
#             else:
#                 continue

#         df_valid_pred_all = pd.concat([df_valid_pred_all, pd.DataFrame([dict_pair])], ignore_index=True)


#     print(" ---------- Evaluation summary statistics: -----------")
#     print(df_valid_pred_all.similarity.describe())



#     SIMILARITY_FILENAME = f'{SIMILARITY_LLM_FILENAME}_{entity_pred}.parquet'
#     SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

#     print("Saving evaluation statistics and scores to ", SIMILARITY_FILEPATH.stem, "[.parquet, _stats.json]")
#     with open(SIMILARITY_FILEPATH, 'w') as f:
#         # results
#         df_valid_pred_all_pyarrow = pa.Table.from_pandas(df_valid_pred_all)
#         pq.write_table(df_valid_pred_all_pyarrow, SIMILARITY_FILEPATH)   
#         #   summary statistics
#         df_valid_pred_all_stats = df_valid_pred_all.describe()
#         f = SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_stats.json"  
#         df_valid_pred_all_stats.to_json(f, indent=4)



#     cos_smlrty_thresh = 0.75
#     df_valid_pred_all['is_similar'] = df_valid_pred_all['similarity'] >= cos_smlrty_thresh
#     print(f"Number of similar impact cases (similarity >= {cos_smlrty_thresh}): {df_valid_pred_all['is_similar'].sum()} out of {len(df_valid_pred_all)}")

#     SIMILARITY_FILENAME = f'{SIMILARITY_LLM_FILENAME}_{entity_pred}_75.parquet'
#     SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

#     with open(SIMILARITY_FILEPATH, 'w') as f:
#         # results
#         df_valid_pred_all_pyarrow = pa.Table.from_pandas(df_valid_pred_all)
#         pq.write_table(df_valid_pred_all_pyarrow, SIMILARITY_FILEPATH)  
#         #   summary statistics
#         df_valid_pred_all_stats = df_valid_pred_all.describe()
#         f = SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_stats.json"  
#         df_valid_pred_all_stats.to_json(f, indent=4)


In [ ]:

# #  Define folder for handling and writing outputs
# def write_to_file(data, out_folder, filename):
#     """Convert output to DataFrame and write to file"""
#     df = pd.DataFrame(list(data), columns=['tag', 'sts_score'])
#     #  Sort the DataFrame by similarity (explicitly)
#     df = df.sort_values(by='sts_score', ascending=False)
#     #  Assign integers to ranking
#     df['rank'] = df['sts_score'].rank(method='first', ascending=False).astype(int)
#     #  Only keep the first 20 resulting tags
#     df = df.head(50)
#     #  Save to file
#     df.to_csv(out_folder / f'{filename}_output.csv', index=False)

# #  Fill run metrics to dictionary
# def handle_metrics(metrics, model_name, length, end_time, start_time):
#     print(f'-> Took {end_time - start_time:.2f} seconds. Number of tags: {length}.')
#     metrics.append({
#         'modelname': model_name,
#         'runtime': round(end_time - start_time, 2),
#         'tagcount': length
#     })
#     return metrics

# class CPU_Unpickler(pickle.Unpickler):
#     """Fix for having issues with loading models on CPU"""
#     def find_class(self, module, name):
#         if module == 'torch.storage' and name == '_load_from_bytes':
#             return lambda b: torch.load(io.BytesIO(b), map_location='cpu')
#         else: return super().find_class(module, name)
